In [95]:
%pip install vnpy
%pip install baostock
import vnpy
import baostock as bs
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import akshare as ak
from zoneinfo import ZoneInfo
from vnpy.trader.object import BarData
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.database import get_database


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [96]:
df = pd.read_parquet("688981_5min_20200716-20260602.parquet")
df.head(100)

,code,open,high,low,close,volume,amount,adjustflag
datetime,,,,,,,,
2020-07-16 09:35:00,688981,95.0000000000,95.0000000000,83.0000000000,86.8600000000,158727622,14381175040.0000,2
2020-07-16 09:40:00,688981,86.0000000000,87.9800000000,80.0000000000,86.0000000000,59510624,5004495104.0000,2
2020-07-16 09:45:00,688981,86.0100000000,88.0000000000,84.9900000000,87.1300000000,36040553,3114025984.0000,2
2020-07-16 09:50:00,688981,87.0000000000,87.3000000000,85.2800000000,85.5000000000,14591281,1265563392.0000,2
2020-07-16 09:55:00,688981,85.5200000000,85.8500000000,85.3300000000,85.3500000000,13968256,1196435200.0000,2
...,...,...,...,...,...,...,...,...
2020-07-17 15:00:00,688981,77.1300000000,77.1700000000,77.0600000000,77.0600000000,4671325,360155904.0000,2
2020-07-20 09:35:00,688981,77.1900000000,77.2000000000,74.0000000000,75.2500000000,20643891,1558470720.0000,2
2020-07-20 09:40:00,688981,75.1100000000,75.1100000000,72.3000000000,72.6400000000,18952217,1392710480.0000,2


In [97]:
df.describe()

,code,open,high,low,close,volume,amount,adjustflag
count,68304,68304,68304,68304,68304,68304,68304,68304
unique,1,9084,9044,8939,9098,66020,67713,1
top,688981,0E-10,0E-10,0E-10,0E-10,0,0.0000,2
freq,68304,288,288,288,288,297,297,68304


In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 68304 entries, 2020-07-16 09:35:00 to 2026-06-01 15:00:00
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   code        68304 non-null  object
 1   open        68304 non-null  object
 2   high        68304 non-null  object
 3   low         68304 non-null  object
 4   close       68304 non-null  object
 5   volume      68304 non-null  object
 6   amount      68304 non-null  object
 7   adjustflag  68304 non-null  object
dtypes: object(8)
memory usage: 4.7+ MB


In [99]:
float_cols = ["open", "high", "low", "close", "volume", "amount"]


# 这个errors的意思是遇到不能转换的格式，就变成NaN，而不是报错。
df[float_cols] = df[float_cols].apply(pd.to_numeric, errors="coerce")

In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 68304 entries, 2020-07-16 09:35:00 to 2026-06-01 15:00:00
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   code        68304 non-null  object 
 1   open        68304 non-null  float64
 2   high        68304 non-null  float64
 3   low         68304 non-null  float64
 4   close       68304 non-null  float64
 5   volume      68304 non-null  int64  
 6   amount      68304 non-null  float64
 7   adjustflag  68304 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 4.7+ MB


In [101]:
# 特征工程-01-处理时间段问题（早晚、周末、节假日）

# =========================
# 交易日历
# =========================

# df = df.copy()
# df = df.sort_index()

# ## 确保索引是 datetime 类型
# if not isinstance(df.index, pd.DatetimeIndex):
#     df.index = pd.to_datetime(df.index)

# ## 构造交易日期
# df["trade_date"] = df.index.normalize()

# ## 构造交易日日历
# trade_dates = pd.Series(df["trade_date"].unique()).sort_values()
# calendar = pd.DataFrame({"trade_date": trade_dates})

# ## 上一个交易日
# calendar["prev_trade_date"] = calendar["trade_date"].shift(1)

# ## 当前交易日距离上一个交易日的自然日间隔
# calendar["gap_days"] = (
#     calendar["trade_date"] - calendar["prev_trade_date"]
# ).dt.days

# ## 是否普通连续交易日
# calendar["is_normal_day"] = (calendar["gap_days"] == 1).astype(int)

# ## 是否周末后
# calendar["is_after_weekend"] = (
#     (calendar["gap_days"] == 3) &
#     (calendar["trade_date"].dt.dayofweek == 0)
# ).astype(int)

# ## 是否长假后
# calendar["is_after_holiday"] = (calendar["gap_days"] > 3).astype(int)

# ## 合并前先删除旧的交易日历特征，避免重复运行时报错
# calendar_feature_cols = [
#     "gap_days",
#     "is_normal_day",
#     "is_after_weekend",
#     "is_after_holiday",
# ]

# df = df.drop(columns=[c for c in calendar_feature_cols if c in df.columns])

# ## 合并交易日历特征
# df = df.join(
#     calendar.set_index("trade_date")[
#         ["gap_days", "is_normal_day", "is_after_weekend", "is_after_holiday"]
#     ],
#     on="trade_date"
# )

# ## 缺失值填充
# df["gap_days"] = df["gap_days"].fillna(1)
# df["is_normal_day"] = df["is_normal_day"].fillna(1)
# df["is_after_weekend"] = df["is_after_weekend"].fillna(0)
# df["is_after_holiday"] = df["is_after_holiday"].fillna(0)

In [102]:
# =========================
# 特征工程
# =========================

# 首先这里重新从 raw_df 开始，避免重复运行 cell 之后产生重复列或者污染
df = df.copy()
df = df.sort_index()

# 如果索引不是时间索引，先转成时间索引
if not isinstance(df.index, pd.DatetimeIndex):
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.set_index("datetime")
    else:
        df.index = pd.to_datetime(df.index)

# 基础参数
EPS = 1e-12

WINDOWS = [1, 3, 5, 10, 20, 48]
SHORT_WINDOWS = [3, 5, 10, 20]
MID_WINDOWS = [5, 10, 20, 48]
LONG_WINDOWS = [20, 48]

# 交易日期
df["trade_date"] = df.index.normalize()

In [103]:
# =========================
# 收益率特征
# =========================

# 这里收益率都是用当前和过去的数据构造，不存在未来函数
df["ret_1"] = df["close"].pct_change(1)
df["log_close"] = np.log(df["close"] + EPS)

for w in WINDOWS:
    df[f"ret_{w}"] = df["close"].pct_change(w)
    df[f"log_ret_{w}"] = df["log_close"].diff(w)

In [104]:
# =========================
# 均线特征
# =========================

# 注意：ma_w 是绝对价格水平，只作为中间变量，不进入模型
# 真正进入模型的是 ma_dev、ma_slope、ma_gap 这种相对特征
for w in MID_WINDOWS:
    df[f"ma_{w}"] = df["close"].rolling(w).mean()
    df[f"ma_dev_{w}"] = df["close"] / (df[f"ma_{w}"] + EPS) - 1
    df[f"ma_slope_{w}"] = df[f"ma_{w}"] / (df[f"ma_{w}"].shift(1) + EPS) - 1

# 多周期均线偏离
df["ma_gap_5_20"] = df["ma_5"] / (df["ma_20"] + EPS) - 1
df["ma_gap_10_48"] = df["ma_10"] / (df["ma_48"] + EPS) - 1

In [105]:
# =========================
# 动量特征
# =========================

# 多周期动量差
df["mom_3_10"] = df["ret_3"] - df["ret_10"]
df["mom_5_20"] = df["ret_5"] - df["ret_20"]
df["mom_10_48"] = df["ret_10"] - df["ret_48"]

# 非重叠动量
df["ret_recent_3"] = df["close"] / (df["close"].shift(3) + EPS) - 1
df["ret_prev_3"] = df["close"].shift(3) / (df["close"].shift(6) + EPS) - 1
df["ret_prev_6"] = df["close"].shift(3) / (df["close"].shift(9) + EPS) - 1

df["mom_change_3"] = df["ret_recent_3"] - df["ret_prev_3"]
df["mom_change_6"] = df["ret_recent_3"] - df["ret_prev_6"]

# 上涨下跌K线比例
df["up_bar"] = (df["close"] > df["close"].shift(1)).astype(int)
df["down_bar"] = (df["close"] < df["close"].shift(1)).astype(int)

for w in SHORT_WINDOWS:
    df[f"up_ratio_{w}"] = df["up_bar"].rolling(w).mean()
    df[f"down_ratio_{w}"] = df["down_bar"].rolling(w).mean()

In [106]:
# =========================
# K线结构特征
# =========================

# K线实体
df["body"] = (df["close"] - df["open"]) / (df["open"] + EPS)

# K线振幅
df["bar_range"] = (df["high"] - df["low"]) / (df["close"] + EPS)

# 上影线和下影线
df["upper_shadow"] = (df["high"] - df[["open", "close"]].max(axis=1)) / (df["close"] + EPS)
df["lower_shadow"] = (df[["open", "close"]].min(axis=1) - df["low"]) / (df["close"] + EPS)

# 收盘价在K线内部的位置
df["close_position"] = (df["close"] - df["low"]) / (df["high"] - df["low"] + EPS)

In [107]:
# =========================
# 波动率特征
# =========================

# 收益率波动率
for w in MID_WINDOWS:
    df[f"vol_{w}"] = df["ret_1"].rolling(w).std()

# 振幅波动率
for w in MID_WINDOWS:
    df[f"range_mean_{w}"] = df["bar_range"].rolling(w).mean()
    df[f"range_vol_{w}"] = df["bar_range"].rolling(w).std()

# Parkinson 波动率
hl = np.log((df["high"] + EPS) / (df["low"] + EPS)) ** 2

for w in MID_WINDOWS:
    df[f"parkinson_vol_{w}"] = np.sqrt(hl.rolling(w).mean() / (4 * np.log(2)))

# 波动率比值
df["vol_ratio_5_20"] = df["vol_5"] / (df["vol_20"] + EPS)
df["vol_ratio_10_48"] = df["vol_10"] / (df["vol_48"] + EPS)

# 波动率标准化动量
df["mom_z_3_20"] = df["ret_3"] / (df["vol_20"] + EPS)
df["mom_z_5_20"] = df["ret_5"] / (df["vol_20"] + EPS)
df["mom_z_10_48"] = df["ret_10"] / (df["vol_48"] + EPS)

# 波动率 zscore
for i in [5, 10, 20]:
    for w in [20, 48]:
        vol_mean = df[f"vol_{i}"].rolling(w).mean()
        vol_std = df[f"vol_{i}"].rolling(w).std()
        df[f"vol_zscore_{i}_{w}"] = (df[f"vol_{i}"] - vol_mean) / (vol_std + EPS)

In [108]:
# =========================
# VWAP 特征
# =========================
df = df.copy()
# 单根K线 VWAP
df["vwap_bar"] = df["amount"] / (df["volume"] + EPS)
df["vwap_bar"] = df["vwap_bar"].fillna(df["close"])

# 单根K线 VWAP 偏离
df["vwap_bar_dev"] = df["close"] / (df["vwap_bar"] + EPS) - 1

# 日内累计 VWAP
df["amount_cum_day"] = df.groupby("trade_date")["amount"].cumsum()
df["volume_cum_day"] = df.groupby("trade_date")["volume"].cumsum()

df["vwap_cum_day"] = df["amount_cum_day"] / (df["volume_cum_day"] + EPS)
df["vwap_cum_day"] = df["vwap_cum_day"].fillna(df["close"])

# 日内 VWAP 偏离
df["vwap_dev_day"] = df["close"] / (df["vwap_cum_day"] + EPS) - 1

# 日内 VWAP 斜率
df["vwap_cum_slope_3"] = df["vwap_cum_day"] / (df["vwap_cum_day"].shift(3) + EPS) - 1
df["vwap_cum_slope_6"] = df["vwap_cum_day"] / (df["vwap_cum_day"].shift(6) + EPS) - 1

# 滚动 VWAP
# 注意：vwap_roll_w 是绝对价格水平，只作为中间变量，不进入模型
for w in MID_WINDOWS:
    amount_roll = df["amount"].rolling(w).sum()
    volume_roll = df["volume"].rolling(w).sum()
    
    df[f"vwap_roll_{w}"] = amount_roll / (volume_roll + EPS)
    df[f"vwap_roll_{w}"] = df[f"vwap_roll_{w}"].fillna(df["close"])
    
    df[f"vwap_roll_dev_{w}"] = df["close"] / (df[f"vwap_roll_{w}"] + EPS) - 1
    df[f"vwap_roll_slope_{w}"] = df[f"vwap_roll_{w}"] / (df[f"vwap_roll_{w}"].shift(1) + EPS) - 1

# 滚动VWAP和日内VWAP的相对差
df["vwap_spread_20_day"] = df["vwap_roll_20"] / (df["vwap_cum_day"] + EPS) - 1
df["vwap_spread_48_day"] = df["vwap_roll_48"] / (df["vwap_cum_day"] + EPS) - 1

In [109]:
# =========================
# CLV 特征
# =========================

# CLV 衡量收盘价在K线区间中的强弱
df["clv"] = ((df["close"] - df["low"]) - (df["high"] - df["close"])) / (df["high"] - df["low"] + EPS)
df["clv"] = df["clv"].clip(-1, 1)

for w in MID_WINDOWS:
    df[f"clv_mean_{w}"] = df["clv"].rolling(w).mean()
    df[f"clv_sum_{w}"] = df["clv"].rolling(w).sum()

# 成交量加权 CLV
df["clv_volume"] = df["clv"] * np.log1p(df["volume"])

for w in MID_WINDOWS:
    df[f"clv_volume_sum_{w}"] = df["clv_volume"].rolling(w).sum()
    df[f"clv_volume_mean_{w}"] = df["clv_volume"].rolling(w).mean()

# ADL rolling normalized
df["money_flow_volume"] = df["clv"] * df["volume"]

for w in [10, 20, 48]:
    mfv_sum = df["money_flow_volume"].rolling(w).sum()
    vol_sum = df["volume"].rolling(w).sum()
    df[f"adl_roll_norm_{w}"] = mfv_sum / (vol_sum + EPS)

In [110]:
# =========================
# 成交量和成交额特征
# =========================

# 注意：volume_ma 和 amount_ma 只作为中间变量，不进入模型
# 真正进入模型的是 ratio / log_ratio / direction 这类相对特征
for w in MID_WINDOWS:
    df[f"volume_ma_{w}"] = df["volume"].rolling(w).mean()
    df[f"amount_ma_{w}"] = df["amount"].rolling(w).mean()
    
    df[f"volume_ratio_{w}"] = df["volume"] / (df[f"volume_ma_{w}"] + EPS)
    df[f"amount_ratio_{w}"] = df["amount"] / (df[f"amount_ma_{w}"] + EPS)
    
    df[f"log_volume_ratio_{w}"] = np.log1p(df["volume"]) - np.log1p(df[f"volume_ma_{w}"])
    df[f"log_amount_ratio_{w}"] = np.log1p(df["amount"]) - np.log1p(df[f"amount_ma_{w}"])

# 成交量变化率
df["volume_change"] = df["volume"] / (df["volume"].shift(1) + EPS) - 1
df["amount_change"] = df["amount"] / (df["amount"].shift(1) + EPS) - 1

# 收益率和成交量结合
df["ret_volume"] = df["ret_1"] * np.log1p(df["volume"])
df["ret_amount"] = df["ret_1"] * np.log1p(df["amount"])

for w in [5, 10, 20]:
    df[f"ret_volume_sum_{w}"] = df["ret_volume"].rolling(w).sum()
    df[f"ret_amount_sum_{w}"] = df["ret_amount"].rolling(w).sum()

In [111]:
# =========================
# 成交量方向确认特征
# =========================
df = df.copy()
# 这里重点解决原来模型只学绝对成交量的问题
# signed_volume / signed_amount 带有方向信息
df["signed_volume"] = np.sign(df["ret_1"]) * np.log1p(df["volume"])
df["signed_amount"] = np.sign(df["ret_1"]) * np.log1p(df["amount"])

for w in [5, 10, 20, 48]:
    df[f"signed_volume_sum_{w}"] = df["signed_volume"].rolling(w).sum()
    df[f"signed_amount_sum_{w}"] = df["signed_amount"].rolling(w).sum()
    df[f"signed_volume_mean_{w}"] = df["signed_volume"].rolling(w).mean()
    df[f"signed_amount_mean_{w}"] = df["signed_amount"].rolling(w).mean()

# 放量上涨 / 放量下跌
df["up_volume_ratio"] = df["up_bar"] * df["volume_ratio_20"]
df["down_volume_ratio"] = df["down_bar"] * df["volume_ratio_20"]

df["up_amount_ratio"] = df["up_bar"] * df["amount_ratio_20"]
df["down_amount_ratio"] = df["down_bar"] * df["amount_ratio_20"]

for w in [5, 10, 20]:
    df[f"up_volume_ratio_mean_{w}"] = df["up_volume_ratio"].rolling(w).mean()
    df[f"down_volume_ratio_mean_{w}"] = df["down_volume_ratio"].rolling(w).mean()
    df[f"up_amount_ratio_mean_{w}"] = df["up_amount_ratio"].rolling(w).mean()
    df[f"down_amount_ratio_mean_{w}"] = df["down_amount_ratio"].rolling(w).mean()

In [112]:
# =========================
# 日内结构特征
# =========================

# bar_in_day 表示当前是当天第几根K线
df["bar_in_day"] = df.groupby("trade_date").cumcount()

# 每天总K线数量
df["bars_in_day"] = df.groupby("trade_date")["close"].transform("count")

# 日内位置百分比
df["bar_in_day_pct"] = df["bar_in_day"] / (df["bars_in_day"] - 1 + EPS)

# 上午 / 下午
df["is_morning"] = (
    (df.index.time >= pd.Timestamp("09:30").time()) &
    (df.index.time <= pd.Timestamp("11:30").time())
).astype(int)

df["is_afternoon"] = (
    (df.index.time >= pd.Timestamp("13:00").time()) &
    (df.index.time <= pd.Timestamp("15:00").time())
).astype(int)

# 开盘30分钟、收盘30分钟、午盘开盘
df["is_open_30min"] = (df["bar_in_day"] < 6).astype(int)
df["is_close_30min"] = (df["bar_in_day"] >= df["bars_in_day"] - 6).astype(int)

df["is_after_lunch_open"] = (
    (df.index.time >= pd.Timestamp("13:00").time()) &
    (df.index.time <= pd.Timestamp("13:30").time())
).astype(int)

In [113]:
# =========================
# 交易日历特征
# =========================

trade_dates = pd.Series(df["trade_date"].unique()).sort_values()

calendar = pd.DataFrame({
    "trade_date": trade_dates
})

calendar["prev_trade_date"] = calendar["trade_date"].shift(1)
calendar["gap_days"] = (calendar["trade_date"] - calendar["prev_trade_date"]).dt.days

calendar["gap_days"] = calendar["gap_days"].fillna(1)

# 连续交易日
calendar["is_normal_day"] = (calendar["gap_days"] == 1).astype(int)

# 周末后第一个交易日
calendar["is_after_weekend"] = (
    (calendar["gap_days"] == 3) &
    (calendar["trade_date"].dt.dayofweek == 0)
).astype(int)

# 长假后第一个交易日
calendar["is_after_holiday"] = (calendar["gap_days"] > 3).astype(int)

# 防止重复运行 join 后同名列冲突
calendar_feature_cols = [
    "gap_days",
    "is_normal_day",
    "is_after_weekend",
    "is_after_holiday"
]

df = df.drop(columns=[c for c in calendar_feature_cols if c in df.columns])

df = df.join(
    calendar.set_index("trade_date")[calendar_feature_cols],
    on="trade_date"
)

df[calendar_feature_cols] = df[calendar_feature_cols].fillna(0)

In [114]:
# =========================
# 同时刻成交量相对特征
# =========================

# 注意：same_bar_mean 是绝对成交量/成交额均值，只作为中间变量，不进入模型
# 使用 shift(1)，避免当前样本进入自身的历史均值
df["volume_same_bar_mean_20d"] = (
    df.groupby("bar_in_day")["volume"]
    .transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
)

df["amount_same_bar_mean_20d"] = (
    df.groupby("bar_in_day")["amount"]
    .transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
)

df["volume_same_bar_ratio"] = df["volume"] / (df["volume_same_bar_mean_20d"] + EPS)
df["amount_same_bar_ratio"] = df["amount"] / (df["amount_same_bar_mean_20d"] + EPS)

df["log_volume_same_bar_ratio"] = np.log1p(df["volume"]) - np.log1p(df["volume_same_bar_mean_20d"])
df["log_amount_same_bar_ratio"] = np.log1p(df["amount"]) - np.log1p(df["amount_same_bar_mean_20d"])

In [115]:
# =========================
# 均值回归特征
# =========================

for w in [10, 20, 48]:
    price_mean = df["close"].rolling(w).mean()
    price_std = df["close"].rolling(w).std()
    df[f"price_zscore_{w}"] = (df["close"] - price_mean) / (price_std + EPS)

# 日内高低点位置
df["cum_high_day"] = df.groupby("trade_date")["high"].cummax()
df["cum_low_day"] = df.groupby("trade_date")["low"].cummin()

df["intraday_range_position"] = (
    (df["close"] - df["cum_low_day"]) /
    (df["cum_high_day"] - df["cum_low_day"] + EPS)
)

In [116]:
# =========================
# 流动性特征
# =========================

# Amihud 流动性冲击
df["amihud"] = df["ret_1"].abs() / (df["amount"] + EPS)

for w in [10, 20, 48]:
    df[f"amihud_{w}"] = df["amihud"].rolling(w).mean()

# 单位成交量对应振幅
df["range_per_volume"] = df["bar_range"] / (df["volume"] + EPS)

for w in [10, 20, 48]:
    df[f"range_per_volume_{w}"] = df["range_per_volume"].rolling(w).mean()

In [117]:
# =========================
# 缺失值和无穷值处理
# =========================

# 先把无穷值替换成缺失值
df = df.replace([np.inf, -np.inf], np.nan)

# 删除因为 rolling / shift 产生的缺失值
df = df.dropna().copy()

# 整理 DataFrame 内存碎片
df = df.copy()

print("特征工程后数据量:", df.shape)

特征工程后数据量: (67710, 230)


In [118]:
# =========================
# 标签构造
# =========================

# 首先这里的标签是预测未来收益，所以一定要用 shift(-HORIZON)
# 注意：特征只能使用当前时点及过去信息，标签可以使用未来信息
# 预测未来3根5分钟K线，也就是未来15分钟
HORIZON = 3

# 交易成本假设
COMMISSION_RATE = 0.001      # 佣金 0.10%
SLIPPAGE_RATE = 0.0005       # 滑点 0.05%
TOTAL_COST = COMMISSION_RATE + SLIPPAGE_RATE

# 标签阈值
# 这里不能设成0，因为日内交易有成本
# 0.002表示未来15分钟收益超过0.2%才认为是有效上涨
LABEL_THRESHOLD = 0.002


# =========================
# 未来收益率
# =========================

## 未来 HORIZON 根K线后的收盘价
df["future_close"] = df["close"].shift(-HORIZON)

## 未来 HORIZON 根K线收益率
df["future_return"] = df["future_close"] / df["close"] - 1

## 未来 HORIZON 根K线对数收益率
df["future_log_return"] = np.log(df["future_close"] / df["close"])


# =========================
# 二分类标签
# =========================

## 简单方向标签
## 只要未来收益率大于阈值，就标记为1，否则为0
df["label"] = (df["future_return"] > LABEL_THRESHOLD).astype(int)

## 成本后方向标签
## 未来收益率必须超过交易成本和额外阈值，才认为值得交易
df["label_cost_adjusted"] = (
    df["future_return"] > (TOTAL_COST + LABEL_THRESHOLD)
).astype(int)


# =========================
# 三分类标签
# =========================

## 三分类标签用于区分上涨、震荡、下跌
## 1 表示显著上涨
## 0 表示震荡
## -1 表示显著下跌
UP_THRESHOLD = 0.002
DOWN_THRESHOLD = -0.002

df["label_3class"] = np.where(
    df["future_return"] > UP_THRESHOLD,
    1,
    np.where(df["future_return"] < DOWN_THRESHOLD, -1, 0)
)


# =========================
# 波动率自适应标签
# =========================

## 用过去20根K线的收益率波动率作为当前风险尺度
## 注意这里用的是过去波动率，不是未来波动率，所以没有未来函数
if "vol_20" in df.columns:
    df["target_vol"] = df["vol_20"]
else:
    df["target_vol"] = df["ret_1"].rolling(20).std()

## 波动率自适应上下边界
## 当前波动越大，标签阈值越高
VOL_MULTIPLIER = 1.5

df["dynamic_up_threshold"] = VOL_MULTIPLIER * df["target_vol"]
df["dynamic_down_threshold"] = -VOL_MULTIPLIER * df["target_vol"]

## 波动率自适应三分类标签
df["label_dynamic_3class"] = np.where(
    df["future_return"] > df["dynamic_up_threshold"],
    1,
    np.where(df["future_return"] < df["dynamic_down_threshold"], -1, 0)
)

## 波动率自适应二分类标签
## 只把显著上涨记为1，其余情况记为0
df["label_dynamic_binary"] = (
    df["future_return"] > df["dynamic_up_threshold"]
).astype(int)


# =========================
# 回测用收益标签
# =========================

## 下一根K线收益率，用于后面构造净值或者回测
df["next_ret_1"] = df["close"].shift(-1) / df["close"] - 1

## 未来3根K线的最大上涨幅度
df["future_max_close"] = (
    df["close"]
    .shift(-1)
    .rolling(HORIZON)
    .max()
    .shift(-(HORIZON - 1))
)

df["future_max_return"] = df["future_max_close"] / df["close"] - 1

## 未来3根K线的最大下跌幅度
df["future_min_close"] = (
    df["close"]
    .shift(-1)
    .rolling(HORIZON)
    .min()
    .shift(-(HORIZON - 1))
)

df["future_min_return"] = df["future_min_close"] / df["close"] - 1


# =========================
# 止盈止损触发标签
# =========================

## 这里用于判断未来HORIZON根K线内是否先达到止盈或者止损
## MVP阶段可以先不用，后面做三重障碍标签时再用
TAKE_PROFIT = 0.003
STOP_LOSS = -0.003

df["hit_take_profit"] = (df["future_max_return"] > TAKE_PROFIT).astype(int)
df["hit_stop_loss"] = (df["future_min_return"] < STOP_LOSS).astype(int)

## 简化版交易成功标签
## 未来先不考虑先后顺序，只要达到止盈且没有明显止损，就认为是成功交易
df["label_trade_success"] = (
    (df["hit_take_profit"] == 1) &
    (df["hit_stop_loss"] == 0)
).astype(int)


# =========================
# 标签检查
# =========================

## 检查二分类标签分布
print("label 分布：")
print(df["label"].value_counts(normalize=True))

print("\nlabel_cost_adjusted 分布：")
print(df["label_cost_adjusted"].value_counts(normalize=True))

print("\nlabel_3class 分布：")
print(df["label_3class"].value_counts(normalize=True))

print("\nlabel_dynamic_3class 分布：")
print(df["label_dynamic_3class"].value_counts(normalize=True))

print("\nlabel_trade_success 分布：")
print(df["label_trade_success"].value_counts(normalize=True))


# =========================
# 缺失值处理
# =========================

## 因为标签用了 shift(-HORIZON)，最后 HORIZON 行没有未来数据，需要删除
df = df.replace([np.inf, -np.inf], np.nan)

## 如果你已经完成所有特征工程和标签构造，可以统一 dropna
df = df.dropna().copy()

## 整理 DataFrame 内存碎片
df = df.copy()

label 分布：
label
0    0.754984
1    0.245016
Name: proportion, dtype: float64

label_cost_adjusted 分布：
label_cost_adjusted
0    0.840467
1    0.159533
Name: proportion, dtype: float64

label_3class 分布：
label_3class
 0    0.469340
-1    0.285645
 1    0.245016
Name: proportion, dtype: float64

label_dynamic_3class 分布：
label_dynamic_3class
 0    0.688259
-1    0.160419
 1    0.151322
Name: proportion, dtype: float64

label_trade_success 分布：
label_trade_success
0    0.769207
1    0.230793
Name: proportion, dtype: float64


c:\Users\79967\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [119]:
# =========================
# 特征选择
# =========================

# 不进入模型的字段
exclude_cols = [
    # 原始价格和成交量水平
    "open",
    "high",
    "low",
    "close",
    "volume",
    "amount",
    
    # 时间和日期字段
    "trade_date",
    
    # 绝对价格水平和中间变量
    "log_close",
    "vwap_bar",
    "vwap_cum_day",
    "amount_cum_day",
    "volume_cum_day",
    "cum_high_day",
    "cum_low_day",
    
    # CLV / ADL 中间变量
    "money_flow_volume",
]

# 如果后面已经构造了标签，也要排除
target_cols = [
    # 核心标签
    "label",
    "label_cost_adjusted",
    "label_3class",
    "label_dynamic_3class",
    "label_dynamic_binary",
    "label_trade_success",
    "label_net_up",
    "label_path_up",
    "label_quality_up",
    "label_down",
    "label_path_down",
    
    # 未来收益
    "future_return",
    "future_log_return",
    "future_max_return",
    "future_min_return",
    "next_ret_1",
    "trade_return",
    
    # 未来价格
    "future_close",
    "future_max_close",
    "future_min_close",
    "future_max_price",
    "future_min_price",
    
    # 止盈止损未来触发结果
    "hit_take_profit",
    "hit_stop_loss",
    
    # 动态标签辅助变量
    "target_vol",
    "dynamic_up_threshold",
    "dynamic_down_threshold",
]

exclude_cols = exclude_cols + [c for c in target_cols if c in df.columns]

# 初步选择数值型特征
feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
    and pd.api.types.is_numeric_dtype(df[c])
]

In [120]:
df.head()

,code,open,high,low,close,volume,amount,adjustflag,trade_date,ret_1,...,label_dynamic_3class,label_dynamic_binary,next_ret_1,future_max_close,future_max_return,future_min_close,future_min_return,hit_take_profit,hit_stop_loss,label_trade_success
datetime,,,,,,,,,,,,,,,,,,,,,
2020-07-23 09:35:00,688981,78.01,79.33,77.90,79.20,8071958,633734960.0,2,2020-07-23,-0.004650,...,0,0,-0.008333,79.50,0.003788,78.54,-0.008333,1,1,0
2020-07-23 09:40:00,688981,79.17,79.24,78.29,78.54,3505787,276230000.0,2,2020-07-23,-0.008333,...,0,0,0.005984,79.50,0.012223,78.77,0.002928,1,0,1
2020-07-23 09:45:00,688981,78.57,79.29,78.50,79.01,3100370,244783104.0,2,2020-07-23,0.005984,...,-1,0,0.006202,79.50,0.006202,77.90,-0.014049,1,1,0
2020-07-23 09:50:00,688981,79.00,79.72,78.96,79.50,3239010,257448176.0,2,2020-07-23,0.006202,...,-1,0,-0.009182,78.77,-0.009182,77.88,-0.020377,0,1,0
2020-07-23 09:55:00,688981,79.49,79.49,78.77,78.77,2561446,202545680.0,2,2020-07-23,-0.009182,...,-1,0,-0.011045,77.90,-0.011045,77.05,-0.021836,0,1,0


In [121]:
# =========================
# 删除绝对水平类特征
# =========================

remove_cols = []

for c in feature_cols:
    
    # 删除绝对成交量均值
    if c.startswith("volume_ma_"):
        remove_cols.append(c)
    
    # 删除绝对成交额均值
    if c.startswith("amount_ma_"):
        remove_cols.append(c)
    
    # 删除同时刻绝对成交量均值
    if c.startswith("volume_same_bar_mean_"):
        remove_cols.append(c)
    
    # 删除同时刻绝对成交额均值
    if c.startswith("amount_same_bar_mean_"):
        remove_cols.append(c)
    
    # 删除绝对均线水平，只保留 ma_dev / ma_slope / ma_gap
    if (
        c.startswith("ma_") and
        not c.startswith("ma_dev_") and
        not c.startswith("ma_slope_") and
        not c.startswith("ma_gap_")
    ):
        remove_cols.append(c)
    
    # 删除绝对 VWAP 水平，只保留 dev / slope / spread
    if (
        c.startswith("vwap_roll_") and
        not c.startswith("vwap_roll_dev_") and
        not c.startswith("vwap_roll_slope_")
    ):
        remove_cols.append(c)

remove_cols = sorted(list(set(remove_cols)))

print("删除绝对水平类特征数量:", len(remove_cols))
print(remove_cols)

feature_cols = [c for c in feature_cols if c not in remove_cols]

删除绝对水平类特征数量: 18
['amount_ma_10', 'amount_ma_20', 'amount_ma_48', 'amount_ma_5', 'amount_same_bar_mean_20d', 'ma_10', 'ma_20', 'ma_48', 'ma_5', 'volume_ma_10', 'volume_ma_20', 'volume_ma_48', 'volume_ma_5', 'volume_same_bar_mean_20d', 'vwap_roll_10', 'vwap_roll_20', 'vwap_roll_48', 'vwap_roll_5']


In [122]:
# =========================
# 删除疑似未来函数字段
# =========================

leak_keywords = [
    "future",
    "label",
    "target",
    "next",
    "hit",
]

leak_cols = [
    c for c in feature_cols
    if any(k in c.lower() for k in leak_keywords)
]

print("疑似未来函数字段：")
print(leak_cols)

# 如果存在疑似未来字段，直接从特征中删除
feature_cols = [c for c in feature_cols if c not in leak_cols]

疑似未来函数字段：
[]


In [123]:
# =========================
# 删除常数特征
# =========================

nunique = df[feature_cols].nunique()
constant_cols = [c for c in feature_cols if nunique[c] <= 1]

print("常数特征数量:", len(constant_cols))
print(constant_cols)

feature_cols = [c for c in feature_cols if c not in constant_cols]

print("最终特征数量:", len(feature_cols))
print(feature_cols)

常数特征数量: 1
['bars_in_day']
最终特征数量: 194
['ret_1', 'log_ret_1', 'ret_3', 'log_ret_3', 'ret_5', 'log_ret_5', 'ret_10', 'log_ret_10', 'ret_20', 'log_ret_20', 'ret_48', 'log_ret_48', 'ma_dev_5', 'ma_slope_5', 'ma_dev_10', 'ma_slope_10', 'ma_dev_20', 'ma_slope_20', 'ma_dev_48', 'ma_slope_48', 'ma_gap_5_20', 'ma_gap_10_48', 'mom_3_10', 'mom_5_20', 'mom_10_48', 'ret_recent_3', 'ret_prev_3', 'ret_prev_6', 'mom_change_3', 'mom_change_6', 'up_bar', 'down_bar', 'up_ratio_3', 'down_ratio_3', 'up_ratio_5', 'down_ratio_5', 'up_ratio_10', 'down_ratio_10', 'up_ratio_20', 'down_ratio_20', 'body', 'bar_range', 'upper_shadow', 'lower_shadow', 'close_position', 'vol_5', 'vol_10', 'vol_20', 'vol_48', 'range_mean_5', 'range_vol_5', 'range_mean_10', 'range_vol_10', 'range_mean_20', 'range_vol_20', 'range_mean_48', 'range_vol_48', 'parkinson_vol_5', 'parkinson_vol_10', 'parkinson_vol_20', 'parkinson_vol_48', 'vol_ratio_5_20', 'vol_ratio_10_48', 'mom_z_3_20', 'mom_z_5_20', 'mom_z_10_48', 'vol_zscore_5_20', 'vol_

In [124]:
# =========================
# 时间序列切分
# =========================

target_col = "label"

df_model = df.copy()
df_model = df_model.sort_index()

n = len(df_model)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
valid_df = df_model.iloc[train_end:valid_end].copy()
test_df = df_model.iloc[valid_end:].copy()

print("训练集：", train_df.index.min(), train_df.index.max(), train_df.shape)
print("验证集：", valid_df.index.min(), valid_df.index.max(), valid_df.shape)
print("测试集：", test_df.index.min(), test_df.index.max(), test_df.shape)

print("训练集标签分布：")
print(train_df[target_col].value_counts(normalize=True))

print("验证集标签分布：")
print(valid_df[target_col].value_counts(normalize=True))

print("测试集标签分布：")
print(test_df[target_col].value_counts(normalize=True))

训练集： 2020-07-23 09:35:00 2024-08-15 10:55:00 (47393, 249)
验证集： 2024-08-15 11:00:00 2025-07-03 14:45:00 (10156, 249)
测试集： 2025-07-03 14:50:00 2026-06-01 14:45:00 (10156, 249)
训练集标签分布：
label
0    0.76659
1    0.23341
Name: proportion, dtype: float64
验证集标签分布：
label
0    0.745077
1    0.254923
Name: proportion, dtype: float64
测试集标签分布：
label
0    0.710713
1    0.289287
Name: proportion, dtype: float64


In [125]:
# =========================
# 用训练集分位数缩尾
# =========================

# 这里不能用全样本分位数，否则会有轻微信息泄露
clip_bounds = {}

for c in feature_cols:
    lower = train_df[c].quantile(0.001)
    upper = train_df[c].quantile(0.999)
    clip_bounds[c] = (lower, upper)

for c in feature_cols:
    lower, upper = clip_bounds[c]
    
    train_df[c] = train_df[c].clip(lower, upper)
    valid_df[c] = valid_df[c].clip(lower, upper)
    test_df[c] = test_df[c].clip(lower, upper)

print("缩尾完成")

缩尾完成


In [126]:
# =========================
# 特征量纲检测：标准化前
# =========================

# 目的：检查是否存在极端量纲、近似常数、异常值比例过高的特征。
# 注意：这里只使用训练集统计，避免测试集信息泄露。

raw_feature_check = train_df[feature_cols].copy()

pre_scale_report = pd.DataFrame({
    "dtype": raw_feature_check.dtypes.astype(str),
    "missing_rate": raw_feature_check.isna().mean(),
    "inf_count": np.isinf(raw_feature_check.replace([np.inf, -np.inf], np.nan)).sum(),
    "mean": raw_feature_check.mean(numeric_only=True),
    "std": raw_feature_check.std(numeric_only=True),
    "min": raw_feature_check.min(numeric_only=True),
    "p01": raw_feature_check.quantile(0.01),
    "p50": raw_feature_check.quantile(0.50),
    "p99": raw_feature_check.quantile(0.99),
    "max": raw_feature_check.max(numeric_only=True),
    "abs_max": raw_feature_check.abs().max(numeric_only=True),
    "nunique": raw_feature_check.nunique(),
})

pre_scale_report["range_p99_p01"] = pre_scale_report["p99"] - pre_scale_report["p01"]
pre_scale_report["cv_abs"] = pre_scale_report["std"] / (pre_scale_report["mean"].abs() + 1e-12)

print("特征数量:", len(feature_cols))
print("缺失率最高的特征:")
display(pre_scale_report.sort_values("missing_rate", ascending=False).head(15))

print("abs_max 最大的特征，也就是量纲/极值最大的特征:")
display(pre_scale_report.sort_values("abs_max", ascending=False).head(20))

print("std 最大的特征:")
display(pre_scale_report.sort_values("std", ascending=False).head(20))

print("近似常数或波动极小的特征:")
display(pre_scale_report.sort_values("std", ascending=True).head(20))

# 简单告警
scale_warn_cols = pre_scale_report.index[
    (pre_scale_report["abs_max"] > 1e6) |
    (pre_scale_report["std"] > 1e4) |
    (pre_scale_report["missing_rate"] > 0.05) |
    (pre_scale_report["nunique"] <= 2)
].tolist()

print("可能需要关注的特征数量:", len(scale_warn_cols))
print(scale_warn_cols[:80])


特征数量: 194
缺失率最高的特征:


,dtype,missing_rate,inf_count,mean,std,min,p01,p50,p99,max,abs_max,nunique,range_p99_p01,cv_abs
ret_1,float64,0.0,0,-0.000003,0.002986,-0.018285,-0.007921,-0.000156,0.009512,0.024601,0.024601,32891,0.017432,867.407854
log_ret_1,float64,0.0,0,-0.000008,0.002982,-0.018454,-0.007952,-0.000156,0.009467,0.024304,0.024304,33047,0.017419,377.809536
ret_3,float64,0.0,0,-0.000008,0.005367,-0.031245,-0.014328,-0.000221,0.018331,0.045753,0.045753,36927,0.032659,672.775597
log_ret_3,float64,0.0,0,-0.000022,0.005349,-0.031743,-0.014431,-0.000221,0.018165,0.044737,0.044737,36986,0.032596,239.701021
ret_5,float64,0.0,0,-0.000019,0.006998,-0.039782,-0.018557,-0.000364,0.023966,0.053338,0.053338,38763,0.042522,370.263545
log_ret_5,float64,0.0,0,-0.000043,0.006972,-0.040595,-0.018731,-0.000364,0.023683,0.051964,0.051964,38872,0.042414,161.153214
ret_10,float64,0.0,0,-0.000049,0.010098,-0.055141,-0.026928,-0.000553,0.034441,0.070896,0.070896,40900,0.061369,207.044966
log_ret_10,float64,0.0,0,-0.000099,0.010054,-0.056720,-0.027297,-0.000553,0.033861,0.068495,0.068495,40959,0.061158,101.091344
ret_20,float64,0.0,0,-0.000130,0.014384,-0.073018,-0.039554,-0.000887,0.047700,0.086982,0.086982,42554,0.087254,110.388666
log_ret_20,float64,0.0,0,-0.000233,0.014328,-0.075822,-0.040358,-0.000887,0.046597,0.083405,0.083405,42640,0.086955,61.435899


abs_max 最大的特征，也就是量纲/极值最大的特征:


,dtype,missing_rate,inf_count,mean,std,min,p01,p50,p99,max,abs_max,nunique,range_p99_p01,cv_abs
signed_amount_sum_48,float64,0.0,0,-45.633451,109.929097,-355.794603,-280.570746,-49.203168,221.242147,312.179113,355.794603,47145,501.812893,2.408959
signed_volume_sum_48,float64,0.0,0,-34.888706,84.902942,-275.267069,-216.787054,-37.534742,171.249649,247.246580,275.267069,47142,388.036704,2.433537
clv_volume_sum_48,float64,0.0,0,-30.851500,68.207980,-237.969409,-182.138140,-33.269668,140.127544,191.813477,237.969409,47284,322.265684,2.210848
signed_amount_sum_20,float64,0.0,0,-18.985645,69.263184,-221.611516,-173.600061,-18.652514,147.766018,211.193146,221.611516,47157,321.366079,3.648187
signed_volume_sum_20,float64,0.0,0,-14.516043,53.428602,-170.375406,-133.921560,-14.708729,114.817390,165.718484,170.375406,47158,248.738951,3.680659
signed_amount_sum_10,float64,0.0,0,-9.484743,48.941195,-149.392621,-115.083924,-2.646362,107.697179,148.631774,149.392621,47169,222.781103,5.159992
clv_volume_sum_20,float64,0.0,0,-12.849229,41.004390,-133.272465,-103.661563,-13.713268,87.682322,121.076953,133.272465,47281,191.343885,3.191194
signed_volume_sum_10,float64,0.0,0,-7.252681,37.707147,-116.446802,-89.225401,-2.649328,83.610688,116.234871,116.446802,47172,172.836089,5.199063
signed_amount_sum_5,float64,0.0,0,-4.741777,34.828730,-94.485989,-85.501999,-15.184499,82.774844,95.040894,95.040894,47135,168.276842,7.345080
clv_volume_sum_10,float64,0.0,0,-6.421234,28.089313,-89.985424,-68.587528,-7.143237,61.672282,82.696993,89.985424,47282,130.259809,4.374442


std 最大的特征:


,dtype,missing_rate,inf_count,mean,std,min,p01,p50,p99,max,abs_max,nunique,range_p99_p01,cv_abs
signed_amount_sum_48,float64,0.0,0,-45.633451,109.929097,-355.794603,-280.570746,-49.203168,221.242147,312.179113,355.794603,47145,501.812893,2.408959
signed_volume_sum_48,float64,0.0,0,-34.888706,84.902942,-275.267069,-216.787054,-37.534742,171.249649,247.246580,275.267069,47142,388.036704,2.433537
signed_amount_sum_20,float64,0.0,0,-18.985645,69.263184,-221.611516,-173.600061,-18.652514,147.766018,211.193146,221.611516,47157,321.366079,3.648187
clv_volume_sum_48,float64,0.0,0,-30.851500,68.207980,-237.969409,-182.138140,-33.269668,140.127544,191.813477,237.969409,47284,322.265684,2.210848
signed_volume_sum_20,float64,0.0,0,-14.516043,53.428602,-170.375406,-133.921560,-14.708729,114.817390,165.718484,170.375406,47158,248.738951,3.680659
signed_amount_sum_10,float64,0.0,0,-9.484743,48.941195,-149.392621,-115.083924,-2.646362,107.697179,148.631774,149.392621,47169,222.781103,5.159992
clv_volume_sum_20,float64,0.0,0,-12.849229,41.004390,-133.272465,-103.661563,-13.713268,87.682322,121.076953,133.272465,47281,191.343885,3.191194
signed_volume_sum_10,float64,0.0,0,-7.252681,37.707147,-116.446802,-89.225401,-2.649328,83.610688,116.234871,116.446802,47172,172.836089,5.199063
signed_amount_sum_5,float64,0.0,0,-4.741777,34.828730,-94.485989,-85.501999,-15.184499,82.774844,95.040894,95.040894,47135,168.276842,7.345080
clv_volume_sum_10,float64,0.0,0,-6.421234,28.089313,-89.985424,-68.587528,-7.143237,61.672282,82.696993,89.985424,47282,130.259809,4.374442


近似常数或波动极小的特征:


,dtype,missing_rate,inf_count,mean,std,min,p01,p50,p99,max,abs_max,nunique,range_p99_p01,cv_abs
amihud_48,float64,0.0,0,8.550982e-11,4.427804e-11,1.731680e-11,2.304673e-11,7.731740e-11,2.431457e-10,3.116351e-10,3.116351e-10,47131,2.200990e-10,0.511827
amihud_20,float64,0.0,0,8.549228e-11,4.902899e-11,1.322135e-11,2.023890e-11,7.419615e-11,2.656078e-10,3.529398e-10,3.529398e-10,47139,2.453689e-10,0.566860
amihud_10,float64,0.0,0,8.548425e-11,5.376349e-11,1.026012e-11,1.756004e-11,7.238697e-11,2.787508e-10,4.171214e-10,4.171214e-10,47163,2.611908e-10,0.621656
amihud,float64,0.0,0,8.536404e-11,8.986632e-11,0.000000e+00,0.000000e+00,5.868016e-11,4.281324e-10,7.588318e-10,7.588318e-10,44769,4.281324e-10,1.040552
range_per_volume_48,float64,0.0,0,7.797299e-09,3.256473e-09,1.472309e-09,2.737617e-09,7.318960e-09,1.815992e-08,2.319505e-08,2.319505e-08,47299,1.542231e-08,0.417588
range_per_volume_20,float64,0.0,0,7.796902e-09,3.682977e-09,1.312225e-09,2.287267e-09,7.143490e-09,2.001726e-08,2.803514e-08,2.803514e-08,47299,1.772999e-08,0.472304
range_per_volume_10,float64,0.0,0,7.795861e-09,4.012420e-09,1.285811e-09,1.997039e-09,7.009456e-09,2.119112e-08,3.058246e-08,3.058246e-08,47299,1.919408e-08,0.514620
range_per_volume,float64,0.0,0,7.791698e-09,5.347956e-09,5.076856e-10,1.063031e-09,6.528043e-09,2.704275e-08,4.325118e-08,4.325118e-08,47299,2.597972e-08,0.686278
ma_slope_48,float64,0.0,0,-1.160925e-05,4.703226e-04,-2.207442e-03,-1.297599e-03,-2.235270e-05,1.440828e-03,2.593985e-03,2.593985e-03,47067,2.738427e-03,40.512736
vwap_roll_slope_48,float64,0.0,0,-1.101139e-05,6.332146e-04,-4.573590e-03,-1.705653e-03,-1.862443e-05,2.132040e-03,6.316536e-03,6.316536e-03,47299,3.837693e-03,57.505400


可能需要关注的特征数量: 10
['up_bar', 'down_bar', 'is_morning', 'is_afternoon', 'is_open_30min', 'is_close_30min', 'is_after_lunch_open', 'is_normal_day', 'is_after_weekend', 'is_after_holiday']


In [127]:
# =========================
# 特征标准化
# =========================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df[target_col]
y_valid = valid_df[target_col]
y_test = test_df[target_col]

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

train_scaled = train_df.copy()
valid_scaled = valid_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_cols] = X_train_scaled
valid_scaled[feature_cols] = X_valid_scaled
test_scaled[feature_cols] = X_test_scaled

print("标准化完成")
print("特征数量:", len(feature_cols))

标准化完成
特征数量: 194


In [128]:
# =========================
# 特征最终检查
# =========================

print("是否存在缺失值：", train_scaled[feature_cols].isna().any().any())
print("是否存在无穷值：", np.isinf(train_scaled[feature_cols]).any().any())

print("训练集特征均值最大值：")
print(train_scaled[feature_cols].mean().abs().sort_values(ascending=False).head(10))

print("训练集特征标准差：")
print(train_scaled[feature_cols].std().sort_values(ascending=False).head(10))

print("最终特征数量:", len(feature_cols))

是否存在缺失值： False
是否存在无穷值： False
训练集特征均值最大值：
vol_48                       3.934050e-16
up_amount_ratio_mean_20      3.742145e-16
amihud_48                    3.070478e-16
down_volume_ratio_mean_20    2.926549e-16
down_amount_ratio_mean_20    2.686668e-16
bar_range                    2.590716e-16
bar_in_day_pct               2.460280e-16
volume_ratio_5               2.425985e-16
range_mean_5                 2.302858e-16
up_ratio_20                  2.158930e-16
dtype: float64
训练集特征标准差：
log_ret_1                    1.000011
range_mean_20                1.000011
down_ratio_5                 1.000011
price_zscore_10              1.000011
is_open_30min                1.000011
down_amount_ratio_mean_10    1.000011
intraday_range_position      1.000011
volume_change                1.000011
log_volume_ratio_10          1.000011
vwap_roll_dev_20             1.000011
dtype: float64
最终特征数量: 194


In [129]:
# =========================
# 特征量纲检测：标准化后
# =========================

scaled_feature_check = train_scaled[feature_cols].copy()

post_scale_report = pd.DataFrame({
    "mean": scaled_feature_check.mean(),
    "std": scaled_feature_check.std(),
    "min": scaled_feature_check.min(),
    "p01": scaled_feature_check.quantile(0.01),
    "p50": scaled_feature_check.quantile(0.50),
    "p99": scaled_feature_check.quantile(0.99),
    "max": scaled_feature_check.max(),
    "abs_max": scaled_feature_check.abs().max(),
    "missing_rate": scaled_feature_check.isna().mean(),
})

post_scale_report["mean_abs"] = post_scale_report["mean"].abs()
post_scale_report["std_dev_from_1"] = (post_scale_report["std"] - 1).abs()

print("标准化后均值绝对值最大的特征:")
display(post_scale_report.sort_values("mean_abs", ascending=False).head(20))

print("标准化后std最偏离1的特征:")
display(post_scale_report.sort_values("std_dev_from_1", ascending=False).head(20))

print("标准化后abs_max最大的特征，可能是极端异常点残留:")
display(post_scale_report.sort_values("abs_max", ascending=False).head(20))

bad_scaled_cols = post_scale_report.index[
    (post_scale_report["mean_abs"] > 1e-6) |
    (post_scale_report["std_dev_from_1"] > 1e-3) |
    (post_scale_report["missing_rate"] > 0) |
    (post_scale_report["abs_max"] > 10)
].tolist()

print("标准化后需要关注的特征数量:", len(bad_scaled_cols))
print(bad_scaled_cols[:80])


标准化后均值绝对值最大的特征:


,mean,std,min,p01,p50,p99,max,abs_max,missing_rate,mean_abs,std_dev_from_1
vol_48,3.934050e-16,1.000011,-1.385909,-1.214210,-0.250302,3.555445,6.628698,6.628698,0.0,3.934050e-16,0.000011
up_amount_ratio_mean_20,-3.742145e-16,1.000011,-1.534434,-1.339647,-0.249602,3.542545,5.105596,5.105596,0.0,3.742145e-16,0.000011
amihud_48,3.070478e-16,1.000011,-1.540125,-1.410716,-0.185024,3.560175,5.106993,5.106993,0.0,3.070478e-16,0.000011
down_volume_ratio_mean_20,-2.926549e-16,1.000011,-1.779066,-1.531348,-0.191276,3.096541,4.826953,4.826953,0.0,2.926549e-16,0.000011
down_amount_ratio_mean_20,2.686668e-16,1.000011,-1.789869,-1.538597,-0.190603,3.084322,4.790742,4.790742,0.0,2.686668e-16,0.000011
bar_range,2.590716e-16,1.000011,-1.069063,-1.012341,-0.268170,3.974535,8.500855,8.500855,0.0,2.590716e-16,0.000011
bar_in_day_pct,-2.460280e-16,1.000011,-1.695819,-1.695819,-0.035688,1.696621,1.696621,1.696621,0.0,2.460280e-16,0.000011
volume_ratio_5,2.425985e-16,1.000011,-1.732057,-1.496470,-0.190350,3.439099,5.717668,5.717668,0.0,2.425985e-16,0.000011
range_mean_5,2.302858e-16,1.000011,-1.325365,-1.215508,-0.253799,3.775669,6.893823,6.893823,0.0,2.302858e-16,0.000011
up_ratio_20,-2.158930e-16,1.000011,-2.822793,-2.343406,0.053529,2.450465,2.929852,2.929852,0.0,2.158930e-16,0.000011


标准化后std最偏离1的特征:


,mean,std,min,p01,p50,p99,max,abs_max,missing_rate,mean_abs,std_dev_from_1
log_ret_1,-7.796135e-18,1.000011,-6.185960,-2.664099,-0.049748,3.177288,8.152901,8.152901,0.0,7.796135e-18,0.000011
range_mean_20,1.007501e-16,1.000011,-1.589795,-1.391042,-0.215030,3.497902,5.958030,5.958030,0.0,1.007501e-16,0.000011
down_ratio_5,1.439286e-16,1.000011,-2.365460,-2.365460,0.466698,2.354803,2.354803,2.365460,0.0,1.439286e-16,0.000011
price_zscore_10,-1.439286e-17,1.000011,-2.243677,-1.910886,-0.072156,2.140714,2.447312,2.447312,0.0,1.439286e-17,0.000011
is_open_30min,1.874071e-20,1.000011,-0.378106,-0.378106,-0.378106,2.644763,2.644763,2.644763,0.0,1.874071e-20,0.000011
down_amount_ratio_mean_10,-5.277384e-17,1.000011,-1.539740,-1.358465,-0.237807,3.345629,5.498562,5.498562,0.0,5.277384e-17,0.000011
intraday_range_position,-1.103453e-16,1.000011,-1.591104,-1.591104,-0.042528,1.829075,1.850177,1.850177,0.0,1.103453e-16,0.000011
volume_change,-2.578722e-17,1.000011,-1.283562,-1.128889,-0.226899,3.951753,9.932935,9.932935,0.0,2.578722e-17,0.000011
log_volume_ratio_10,2.368826e-17,1.000011,-2.839446,-2.199094,-0.040166,2.554856,3.532305,3.532305,0.0,2.368826e-17,0.000011
vwap_roll_dev_20,-2.398811e-17,1.000011,-4.854057,-2.713240,-0.032708,3.081145,5.862881,5.862881,0.0,2.398811e-17,0.000011


标准化后abs_max最大的特征，可能是极端异常点残留:


,mean,std,min,p01,p50,p99,max,abs_max,missing_rate,mean_abs,std_dev_from_1
amount_same_bar_ratio,8.635719e-17,1.000011,-0.778730,-0.723189,-0.262758,4.118549,12.435319,12.435319,0.0,8.635719e-17,0.000011
volume_same_bar_ratio,-4.797621e-17,1.000011,-0.804715,-0.746787,-0.263361,4.055704,12.345191,12.345191,0.0,4.797621e-17,0.000011
vwap_cum_slope_3,0.000000e+00,1.000011,-9.736287,-3.512357,-0.007268,3.113764,10.335545,10.335545,0.0,0.000000e+00,0.000011
amount_change,2.758632e-17,1.000011,-1.278130,-1.125083,-0.226481,3.927949,10.162084,10.162084,0.0,2.758632e-17,0.000011
vwap_roll_slope_48,-9.595243e-18,1.000011,-7.205499,-2.676280,-0.012023,3.384435,9.992843,9.992843,0.0,9.595243e-18,0.000011
vwap_roll_slope_20,1.199405e-18,1.000011,-6.313579,-2.600092,-0.033954,3.295137,9.979607,9.979607,0.0,1.199405e-18,0.000011
volume_change,-2.578722e-17,1.000011,-1.283562,-1.128889,-0.226899,3.951753,9.932935,9.932935,0.0,2.578722e-17,0.000011
vol_5,-3.598216e-17,1.000011,-1.045628,-0.967616,-0.268552,3.974383,9.740689,9.740689,0.0,3.598216e-17,0.000011
volume_ratio_48,5.742153e-17,1.000011,-1.004572,-0.914456,-0.294152,4.110319,9.563575,9.563575,0.0,5.742153e-17,0.000011
vwap_roll_slope_10,1.559227e-17,1.000011,-6.255454,-2.602328,-0.038500,3.338214,9.548201,9.548201,0.0,1.559227e-17,0.000011


标准化后需要关注的特征数量: 4
['vwap_cum_slope_3', 'amount_change', 'volume_same_bar_ratio', 'amount_same_bar_ratio']


#### 下一步就是构造其他的特征了，gpt说不用构造过于复杂的盘口特征，容易过拟合
##### 基础特征
    - 收益率
    - 动量
    - 反转
    - K线结构
    - 价格位置
    - 突破/回撤
##### 均值回归系列
    - vwap相关特征
##### z-score偏移
##### 波动率和风险状态
##### 成交量与流动性特征
##### 日内季节性特征（我没有开盘和收盘的数据）
##### 市场与行业联动（beta）

In [130]:
# # 首先这里的 volume 和 amount 不是累计的数值，所以不需要进行做差处理
# df = df.copy()
# df = df.sort_index()

# # 防止除以0
# EPS = 1e-12

# # 确保 trade_date 存在
# if "trade_date" not in df.columns:
#     df["trade_date"] = df.index.normalize()

# # 建议窗口
# WINDOWS = [1, 3, 5, 10, 20, 48]
# SHORT_WINDOWS = [3, 5, 10, 20]
# MID_WINDOWS = [5, 10, 20, 48]
# LONG_WINDOWS = [20, 48]


# # =========================
# # 收益率
# # =========================

# # 普通收益率
# for w in WINDOWS:
#     df[f"ret_{w}"] = df["close"].pct_change(w)

# # 对数收益率
# df["log_close"] = np.log(df["close"] + EPS)
# for w in WINDOWS:
#     df[f"log_ret_{w}"] = df["log_close"] - df["log_close"].shift(w)


# # =========================
# # MA
# # =========================

# # 均线
# for w in MID_WINDOWS:
#     df[f"ma_{w}"] = df["close"].rolling(w).mean()

# # 均线偏离
# for w in MID_WINDOWS:
#     df[f"ma_dev_{w}"] = df["close"] / (df[f"ma_{w}"] + EPS) - 1

# # 均线斜率
# for w in MID_WINDOWS:
#     df[f"ma_slope_{w}"] = df[f"ma_{w}"].pct_change(3)

# # 均线差
# df["ma_gap_5_20"] = df["ma_5"] / (df["ma_20"] + EPS) - 1
# df["ma_gap_10_48"] = df["ma_10"] / (df["ma_48"] + EPS) - 1


# # =========================
# # 动量
# # =========================

# # 多周期动量差
# df["mom_3_10"] = df["ret_3"] - df["ret_10"]
# df["mom_5_20"] = df["ret_5"] - df["ret_20"]
# df["mom_10_48"] = df["ret_10"] - df["ret_48"]

# # 非重叠动量
# df["ret_recent_3"] = df["close"] / (df["close"].shift(3) + EPS) - 1
# df["ret_prev_3"] = df["close"].shift(3) / (df["close"].shift(6) + EPS) - 1
# df["ret_prev_6"] = df["close"].shift(3) / (df["close"].shift(9) + EPS) - 1

# df["mom_change_3"] = df["ret_recent_3"] - df["ret_prev_3"]
# df["mom_change_6"] = df["ret_recent_3"] - df["ret_prev_6"]

# # 上涨和下跌K线比例
# df["up_bar"] = (df["close"] > df["close"].shift(1)).astype(int)
# df["down_bar"] = (df["close"] < df["close"].shift(1)).astype(int)

# for w in SHORT_WINDOWS:
#     df[f"up_ratio_{w}"] = df["up_bar"].rolling(w).mean()
#     df[f"down_ratio_{w}"] = df["down_bar"].rolling(w).mean()


# # =========================
# # K线结构
# # =========================

# # K线实体
# df["body"] = (df["close"] - df["open"]) / (df["open"] + EPS)

# # K线振幅
# df["bar_range"] = (df["high"] - df["low"]) / (df["close"] + EPS)

# # 上影线
# df["upper_shadow"] = (
#     df["high"] - df[["open", "close"]].max(axis=1)
# ) / (df["close"] + EPS)

# # 下影线
# df["lower_shadow"] = (
#     df[["open", "close"]].min(axis=1) - df["low"]
# ) / (df["close"] + EPS)

# # 收盘价在K线区间中的位置，取值在0到1之间
# df["close_position"] = (
#     df["close"] - df["low"]
# ) / (df["high"] - df["low"] + EPS)

# df["close_position"] = df["close_position"].clip(0, 1)


# # =========================
# # 波动率
# # =========================

# ## 收益率波动率
# for w in MID_WINDOWS:
#     df[f"vol_{w}"] = df["ret_1"].rolling(w).std()

# ## 振幅波动率
# for w in MID_WINDOWS:
#     df[f"range_mean_{w}"] = df["bar_range"].rolling(w).mean()
#     df[f"range_vol_{w}"] = df["bar_range"].rolling(w).std()

# ## Parkinson 波动率
# hl = np.log((df["high"] + EPS) / (df["low"] + EPS)) ** 2

# for w in MID_WINDOWS:
#     df[f"parkinson_vol_{w}"] = np.sqrt(
#         hl.rolling(w).mean() / (4 * np.log(2))
#     )

# ## 波动率比值
# df["vol_ratio_5_20"] = df["vol_5"] / (df["vol_20"] + EPS)
# df["vol_ratio_10_48"] = df["vol_10"] / (df["vol_48"] + EPS)

# ## Z-score 标准化的波动率
# for i in [5, 10, 20]:
#     for w in LONG_WINDOWS:
#         vol_mean = df[f"vol_{i}"].rolling(w).mean()
#         vol_std = df[f"vol_{i}"].rolling(w).std()
#         df[f"vol_zscore_{i}_{w}"] = (
#             df[f"vol_{i}"] - vol_mean
#         ) / (vol_std + EPS)

# ## 波动率标准化动量
# df["mom_z_3_20"] = df["ret_3"] / (df["vol_20"] + EPS)
# df["mom_z_5_20"] = df["ret_5"] / (df["vol_20"] + EPS)
# df["mom_z_10_48"] = df["ret_10"] / (df["vol_48"] + EPS)


# # =========================
# # VWAP 成交量加权价格
# # =========================

# ## 单根5min K线的 VWAP
# df["vwap_bar"] = df["amount"] / df["volume"].replace(0, np.nan)
# df["vwap_bar"] = df["vwap_bar"].fillna(df["close"])

# ## 单根5min K线的 VWAP 偏离
# df["vwap_bar_dev"] = df["close"] / (df["vwap_bar"] + EPS) - 1

# ## 日内累计 VWAP
# df["amount_cum_day"] = df.groupby("trade_date")["amount"].cumsum()
# df["volume_cum_day"] = df.groupby("trade_date")["volume"].cumsum()

# df["vwap_cum_day"] = (
#     df["amount_cum_day"] / df["volume_cum_day"].replace(0, np.nan)
# )

# df["vwap_cum_day"] = df["vwap_cum_day"].fillna(df["close"])

# ## 价格偏离日内累计 VWAP
# df["vwap_dev_day"] = df["close"] / (df["vwap_cum_day"] + EPS) - 1

# ## 日内累计 VWAP 斜率
# df["vwap_cum_slope_3"] = df["vwap_cum_day"].pct_change(3)
# df["vwap_cum_slope_6"] = df["vwap_cum_day"].pct_change(6)

# ## 滚动 VWAP
# for w in MID_WINDOWS:
#     amount_roll = df["amount"].rolling(w).sum()
#     volume_roll = df["volume"].rolling(w).sum()

#     df[f"vwap_roll_{w}"] = amount_roll / volume_roll.replace(0, np.nan)
#     df[f"vwap_roll_{w}"] = df[f"vwap_roll_{w}"].fillna(df["close"])

#     df[f"vwap_roll_dev_{w}"] = (
#         df["close"] / (df[f"vwap_roll_{w}"] + EPS) - 1
#     )

#     df[f"vwap_roll_slope_{w}"] = df[f"vwap_roll_{w}"].pct_change(3)

# ## 短期 VWAP 和日内 VWAP 的差
# df["vwap_spread_20_day"] = df["vwap_roll_20"] / (df["vwap_cum_day"] + EPS) - 1
# df["vwap_spread_48_day"] = df["vwap_roll_48"] / (df["vwap_cum_day"] + EPS) - 1


# # =========================
# # CLV 收盘价位置
# # =========================

# ## 单根K线 CLV，取值范围大致在 -1 到 1
# df["clv"] = (
#     (df["close"] - df["low"]) - (df["high"] - df["close"])
# ) / (df["high"] - df["low"] + EPS)

# df["clv"] = df["clv"].clip(-1, 1)

# ## 滚动 CLV 均值
# for w in MID_WINDOWS:
#     df[f"clv_mean_{w}"] = df["clv"].rolling(w).mean()

# ## 滚动 CLV 累计
# for w in MID_WINDOWS:
#     df[f"clv_sum_{w}"] = df["clv"].rolling(w).sum()

# ## 成交量加权 CLV
# df["clv_volume"] = df["clv"] * np.log1p(df["volume"])

# for w in MID_WINDOWS:
#     df[f"clv_volume_sum_{w}"] = df["clv_volume"].rolling(w).sum()
#     df[f"clv_volume_mean_{w}"] = df["clv_volume"].rolling(w).mean()

# ## 归一化 ADL，等价于成交量加权的 CLV
# df["money_flow_volume"] = df["clv"] * df["volume"]

# for w in [10, 20, 48]:
#     mfv_sum = df["money_flow_volume"].rolling(w).sum()
#     vol_sum = df["volume"].rolling(w).sum()

#     df[f"adl_roll_norm_{w}"] = mfv_sum / (vol_sum + EPS)


# # =========================
# # 成交量和成交额
# # =========================

# ## 成交量均值和成交量放大比例
# for w in MID_WINDOWS:
#     df[f"volume_ma_{w}"] = df["volume"].rolling(w).mean()
#     df[f"volume_ratio_{w}"] = (
#         df["volume"] / (df[f"volume_ma_{w}"] + EPS) - 1
#     )

# ## 成交额均值和成交额放大比例
# for w in MID_WINDOWS:
#     df[f"amount_ma_{w}"] = df["amount"].rolling(w).mean()
#     df[f"amount_ratio_{w}"] = (
#         df["amount"] / (df[f"amount_ma_{w}"] + EPS) - 1
#     )

# ## 量价结合
# df["ret_volume"] = df["ret_1"] * np.log1p(df["volume"])

# for w in [5, 10, 20]:
#     df[f"ret_volume_sum_{w}"] = df["ret_volume"].rolling(w).sum()

# ## 成交量变化率
# for w in [1, 3, 5, 10]:
#     df[f"volume_change_{w}"] = df["volume"].pct_change(w)
#     df[f"amount_change_{w}"] = df["amount"].pct_change(w)


# # =========================
# # 日内结构
# # =========================

# ## 当前是当天第几根K线
# df["bar_in_day"] = df.groupby("trade_date").cumcount()

# ## 当天总共有多少根K线
# df["bars_in_day"] = df.groupby("trade_date")["bar_in_day"].transform("max") + 1

# ## 当前处于当天交易进度的百分比
# df["bar_in_day_pct"] = df["bar_in_day"] / (df["bars_in_day"] + EPS)

# ## 是否开盘前30分钟
# df["is_open_30min"] = (df["bar_in_day"] < 6).astype(int)

# ## 是否收盘前30分钟
# df["is_close_30min"] = (
#     df["bars_in_day"] - df["bar_in_day"] <= 6
# ).astype(int)

# ## 是否午后开盘前30分钟
# df["is_after_lunch_open"] = (
#     (df.index.time >= pd.Timestamp("13:00").time()) &
#     (df.index.time <= pd.Timestamp("13:30").time())
# ).astype(int)

# ## 上午和下午
# df["is_morning"] = (
#     (df.index.time >= pd.Timestamp("09:30").time()) &
#     (df.index.time <= pd.Timestamp("11:30").time())
# ).astype(int)

# df["is_afternoon"] = (
#     (df.index.time >= pd.Timestamp("13:00").time()) &
#     (df.index.time <= pd.Timestamp("15:00").time())
# ).astype(int)


# # =========================
# # 交易日历
# # =========================

# ## 构造交易日日历
# trade_dates = pd.Series(df["trade_date"].unique()).sort_values()
# calendar = pd.DataFrame({"trade_date": trade_dates})

# ## 上一个交易日
# calendar["prev_trade_date"] = calendar["trade_date"].shift(1)

# ## 当前交易日距离上一个交易日的自然日间隔
# calendar["gap_days"] = (
#     calendar["trade_date"] - calendar["prev_trade_date"]
# ).dt.days

# ## 是否普通连续交易日
# calendar["is_normal_day"] = (calendar["gap_days"] == 1).astype(int)

# ## 是否周末后
# calendar["is_after_weekend"] = (
#     (calendar["gap_days"] == 3) &
#     (calendar["trade_date"].dt.dayofweek == 0)
# ).astype(int)

# ## 是否长假后
# calendar["is_after_holiday"] = (calendar["gap_days"] > 3).astype(int)


# ## 缺失值填充
# df["gap_days"] = df["gap_days"].fillna(1)
# df["is_normal_day"] = df["is_normal_day"].fillna(1)
# df["is_after_weekend"] = df["is_after_weekend"].fillna(0)
# df["is_after_holiday"] = df["is_after_holiday"].fillna(0)


# # =========================
# # 同时刻成交量相对特征
# # =========================

# ## 当前bar的成交量相对于过去20个交易日同一bar位置的均值
# df["volume_same_bar_mean_20d"] = (
#     df.groupby("bar_in_day")["volume"]
#       .transform(lambda x: x.shift(1).rolling(20).mean())
# )

# df["volume_same_bar_ratio"] = (
#     df["volume"] / (df["volume_same_bar_mean_20d"] + EPS) - 1
# )

# ## 当前bar的成交额相对于过去20个交易日同一bar位置的均值
# df["amount_same_bar_mean_20d"] = (
#     df.groupby("bar_in_day")["amount"]
#       .transform(lambda x: x.shift(1).rolling(20).mean())
# )

# df["amount_same_bar_ratio"] = (
#     df["amount"] / (df["amount_same_bar_mean_20d"] + EPS) - 1
# )


# # =========================
# # 均值回归
# # =========================

# ## 价格 Z-score
# for w in [10, 20, 48]:
#     ma = df["close"].rolling(w).mean()
#     std = df["close"].rolling(w).std()
#     df[f"price_zscore_{w}"] = (df["close"] - ma) / (std + EPS)

# ## 价格在日内累计高低点中的位置
# df["cum_high_day"] = df.groupby("trade_date")["high"].cummax()
# df["cum_low_day"] = df.groupby("trade_date")["low"].cummin()

# df["intraday_range_position"] = (
#     df["close"] - df["cum_low_day"]
# ) / (df["cum_high_day"] - df["cum_low_day"] + EPS)

# df["intraday_range_position"] = df["intraday_range_position"].clip(0, 1)


# # =========================
# # 流动性和冲击成本
# # =========================

# ## Amihud 非流动性
# df["amihud"] = df["ret_1"].abs() / (df["amount"] + EPS)

# for w in [10, 20, 48]:
#     df[f"amihud_{w}"] = df["amihud"].rolling(w).mean()

# ## 单位成交量对应的价格振幅
# df["range_per_volume"] = df["bar_range"] / (np.log1p(df["volume"]) + EPS)

# for w in [10, 20, 48]:
#     df[f"range_per_volume_{w}"] = df["range_per_volume"].rolling(w).mean()

# df = df.copy()

# # =========================
# # 缺失值和无穷值处理
# # =========================

# ## 替换无穷值
# df = df.replace([np.inf, -np.inf], np.nan)

# ## 删除前面因为 rolling 产生的缺失值
# df = df.dropna().copy()

In [131]:
df.head()

,code,open,high,low,close,volume,amount,adjustflag,trade_date,ret_1,...,label_dynamic_3class,label_dynamic_binary,next_ret_1,future_max_close,future_max_return,future_min_close,future_min_return,hit_take_profit,hit_stop_loss,label_trade_success
datetime,,,,,,,,,,,,,,,,,,,,,
2020-07-23 09:35:00,688981,78.01,79.33,77.90,79.20,8071958,633734960.0,2,2020-07-23,-0.004650,...,0,0,-0.008333,79.50,0.003788,78.54,-0.008333,1,1,0
2020-07-23 09:40:00,688981,79.17,79.24,78.29,78.54,3505787,276230000.0,2,2020-07-23,-0.008333,...,0,0,0.005984,79.50,0.012223,78.77,0.002928,1,0,1
2020-07-23 09:45:00,688981,78.57,79.29,78.50,79.01,3100370,244783104.0,2,2020-07-23,0.005984,...,-1,0,0.006202,79.50,0.006202,77.90,-0.014049,1,1,0
2020-07-23 09:50:00,688981,79.00,79.72,78.96,79.50,3239010,257448176.0,2,2020-07-23,0.006202,...,-1,0,-0.009182,78.77,-0.009182,77.88,-0.020377,0,1,0
2020-07-23 09:55:00,688981,79.49,79.49,78.77,78.77,2561446,202545680.0,2,2020-07-23,-0.009182,...,-1,0,-0.011045,77.90,-0.011045,77.05,-0.021836,0,1,0


In [132]:
# =========================
# LSTM 序列数据和 DataLoader
# =========================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# 关键：每次 feature_cols 改动后，都必须重新构造序列和 DataLoader。
# 否则 train_loader 可能还保留旧的 212 维输入。
LOOKBACK = 32

def make_sequence_data(data, feature_cols, target_col, lookback):
    X = data[feature_cols].values
    y = data[target_col].values

    X_seq = []
    y_seq = []
    index_seq = []

    for i in range(lookback, len(data)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])
        index_seq.append(data.index[i])

    return np.array(X_seq), np.array(y_seq), np.array(index_seq)

X_train_seq, y_train_seq, train_index_seq = make_sequence_data(
    train_scaled, feature_cols, target_col, LOOKBACK
)
X_valid_seq, y_valid_seq, valid_index_seq = make_sequence_data(
    valid_scaled, feature_cols, target_col, LOOKBACK
)
X_test_seq, y_test_seq, test_index_seq = make_sequence_data(
    test_scaled, feature_cols, target_col, LOOKBACK
)

print("len(feature_cols):", len(feature_cols))
print("X_train_seq:", X_train_seq.shape)
print("X_valid_seq:", X_valid_seq.shape)
print("X_test_seq:", X_test_seq.shape)

assert X_train_seq.shape[2] == len(feature_cols), (X_train_seq.shape, len(feature_cols))
assert X_valid_seq.shape[2] == len(feature_cols), (X_valid_seq.shape, len(feature_cols))
assert X_test_seq.shape[2] == len(feature_cols), (X_test_seq.shape, len(feature_cols))

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    SeqDataset(X_train_seq, y_train_seq),
    batch_size=128,
    shuffle=False
)

valid_loader = DataLoader(
    SeqDataset(X_valid_seq, y_valid_seq),
    batch_size=256,
    shuffle=False
)

test_loader = DataLoader(
    SeqDataset(X_test_seq, y_test_seq),
    batch_size=256,
    shuffle=False
)

# 再从 loader 中取一个 batch 检查，确保不是旧数据。
X_batch_check, y_batch_check = next(iter(train_loader))
print("batch feature dim:", X_batch_check.shape[-1])
assert X_batch_check.shape[-1] == len(feature_cols)


len(feature_cols): 194
X_train_seq: (47361, 32, 194)
X_valid_seq: (10124, 32, 194)
X_test_seq: (10124, 32, 194)
batch feature dim: 194


In [133]:
# =========================
# 序列输入主导性检测
# =========================

# 目的：检查进入 LSTM 的三维输入中，是否仍有某些特征数值尺度异常大。
# 期望：标准化后，每个特征在序列输入中的 std 接近 1，abs_max 不应长期极端偏大。

flat_train_seq = X_train_seq.reshape(-1, X_train_seq.shape[-1])

seq_scale_report = pd.DataFrame({
    "feature": feature_cols,
    "seq_mean": flat_train_seq.mean(axis=0),
    "seq_std": flat_train_seq.std(axis=0),
    "seq_abs_mean": np.abs(flat_train_seq).mean(axis=0),
    "seq_abs_max": np.abs(flat_train_seq).max(axis=0),
})

print("序列输入 abs_mean 最大的特征:")
display(seq_scale_report.sort_values("seq_abs_mean", ascending=False).head(20))

print("序列输入 abs_max 最大的特征:")
display(seq_scale_report.sort_values("seq_abs_max", ascending=False).head(20))

print("序列输入 std 最偏离1的特征:")
seq_scale_report["std_dev_from_1"] = np.abs(seq_scale_report["seq_std"] - 1)
display(seq_scale_report.sort_values("std_dev_from_1", ascending=False).head(20))

seq_warn = seq_scale_report[
    (seq_scale_report["seq_abs_max"] > 10) |
    (seq_scale_report["std_dev_from_1"] > 0.2)
]
print("序列输入疑似异常特征数量:", len(seq_warn))
display(seq_warn.head(50))


序列输入 abs_mean 最大的特征:


,feature,seq_mean,seq_std,seq_abs_mean,seq_abs_max
170,is_afternoon,0.000480,1.000000,1.000000,1.000359
169,is_morning,-0.000480,1.000000,1.000000,1.000359
31,down_bar,-0.000008,1.000000,0.999997,1.002260
30,up_bar,-0.000002,1.000000,0.993815,1.117965
134,signed_amount,0.000014,0.999948,0.972347,1.290613
133,signed_volume,0.000015,0.999943,0.971671,1.337607
44,close_position,0.000006,0.999967,0.880924,1.546554
86,clv,0.000006,0.999967,0.880924,1.546554
95,clv_volume,0.000005,0.999913,0.878680,1.803383
185,intraday_range_position,-0.000242,0.999897,0.873712,1.850177


序列输入 abs_max 最大的特征:


,feature,seq_mean,seq_std,seq_abs_mean,seq_abs_max
179,amount_same_bar_ratio,0.000155,1.000242,0.535996,12.435319
178,volume_same_bar_ratio,0.000155,1.000236,0.544977,12.345191
74,vwap_cum_slope_3,0.000318,0.999647,0.360317,10.335545
124,amount_change,-0.000061,0.999922,0.634162,10.162084
83,vwap_roll_slope_48,0.000275,0.999989,0.547400,9.992843
81,vwap_roll_slope_20,0.000405,0.999756,0.576168,9.979607
123,volume_change,-0.000064,0.999912,0.636070,9.932935
45,vol_5,-0.000526,0.999716,0.635398,9.740689
119,volume_ratio_48,-0.000399,0.999797,0.638867,9.563575
79,vwap_roll_slope_10,0.000341,0.999631,0.589058,9.548201


序列输入 std 最偏离1的特征:


,feature,seq_mean,seq_std,seq_abs_mean,seq_abs_max,std_dev_from_1
51,range_mean_10,-0.000975,0.998732,0.726532,6.413200,0.001268
49,range_mean_5,-0.000939,0.998833,0.711096,6.893823,0.001167
53,range_mean_20,-0.000829,0.998952,0.735396,5.958030,0.001048
55,range_mean_48,-0.000749,0.999015,0.736005,6.036875,0.000985
41,bar_range,-0.000781,0.999023,0.666828,8.500855,0.000977
57,parkinson_vol_5,-0.000869,0.999045,0.698699,7.365928,0.000955
72,vwap_bar_dev,0.000004,0.999046,0.657345,6.556486,0.000954
43,lower_shadow,-0.000450,0.999060,0.667057,8.867877,0.000940
40,body,0.000023,0.999090,0.648595,7.151018,0.000910
58,parkinson_vol_10,-0.000866,0.999093,0.712118,7.335749,0.000907


序列输入疑似异常特征数量: 4


,feature,seq_mean,seq_std,seq_abs_mean,seq_abs_max,std_dev_from_1
74,vwap_cum_slope_3,0.000318,0.999647,0.360317,10.335545,0.000353
124,amount_change,-0.000061,0.999922,0.634162,10.162084,0.000078
178,volume_same_bar_ratio,0.000155,1.000236,0.544977,12.345191,0.000236
179,amount_same_bar_ratio,0.000155,1.000242,0.535996,12.435319,0.000242


In [134]:
# =========================
# LSTM 模型
# =========================

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.1):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param.data)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param.data)
            elif "bias" in name:
                nn.init.constant_(param.data, 0)
        
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.constant_(self.fc.bias, 0)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden).squeeze(-1)
        return logits

In [135]:
# =========================
# 训练 LSTM
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"

# 训练前再次检查，避免旧 DataLoader / 旧序列混入。
X_batch_check, _ = next(iter(train_loader))
print("len(feature_cols):", len(feature_cols))
print("loader feature dim:", X_batch_check.shape[-1])
assert X_batch_check.shape[-1] == len(feature_cols), "DataLoader 特征维度和 feature_cols 不一致，请重新运行序列构造 cell"

model = LSTMClassifier(
    input_dim=len(feature_cols),
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

print("model input_size:", model.lstm.input_size)
assert model.lstm.input_size == X_batch_check.shape[-1]

## 用训练集正样本比例初始化最后一层 bias，防止概率一开始就乱漂
pos_rate = np.mean(y_train_seq)
pos_rate = np.clip(pos_rate, 1e-6, 1 - 1e-6)
init_bias = np.log(pos_rate / (1 - pos_rate))

with torch.no_grad():
    model.fc.bias.fill_(init_bias)

print("训练集正样本比例:", pos_rate)
print("fc bias 初始化:", init_bias)

## 第一版先不用 pos_weight，先观察模型是否还会概率塌缩
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-6
)


len(feature_cols): 194
loader feature dim: 194
model input_size: 194
训练集正样本比例: 0.23329321593716348
fc bias 初始化: -1.1898083390231267


In [136]:
# =========================
# 预测函数
# =========================

def predict_loader(model, loader):
    model.eval()
    preds = []
    labels = []
    logits_all = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            prob = torch.sigmoid(logits).cpu().numpy()
            
            preds.extend(prob)
            labels.extend(y_batch.numpy())
            logits_all.extend(logits.cpu().numpy())
    
    return np.array(preds), np.array(labels), np.array(logits_all)

In [137]:
# =========================
# 训练循环
# =========================
import copy

best_auc = -np.inf
best_state = None
patience = 10
wait = 0

for epoch in range(100):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        
        ## 防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    train_prob, train_label, train_logits = predict_loader(model, train_loader)
    valid_prob, valid_label, valid_logits = predict_loader(model, valid_loader)
    
    train_auc = roc_auc_score(train_label, train_prob)
    valid_auc = roc_auc_score(valid_label, valid_prob)
    
    print(
        f"Epoch {epoch+1}, "
        f"Loss={total_loss:.4f}, "
        f"Train AUC={train_auc:.4f}, "
        f"Valid AUC={valid_auc:.4f}, "
        f"Train Prob Mean={train_prob.mean():.4f}, "
        f"Valid Prob Mean={valid_prob.mean():.4f}, "
        f"Valid Prob Max={valid_prob.max():.4f}"
    )
    
    if valid_auc > best_auc:
        best_auc = valid_auc
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
    
    if wait >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

Epoch 1, Loss=195.9193, Train AUC=0.6471, Valid AUC=0.6456, Train Prob Mean=0.2500, Valid Prob Mean=0.2666, Valid Prob Max=0.4514
Epoch 2, Loss=193.3450, Train AUC=0.6571, Valid AUC=0.6436, Train Prob Mean=0.2503, Valid Prob Mean=0.2655, Valid Prob Max=0.4541
Epoch 3, Loss=192.1860, Train AUC=0.6699, Valid AUC=0.6422, Train Prob Mean=0.2516, Valid Prob Mean=0.2666, Valid Prob Max=0.5067
Epoch 4, Loss=190.5293, Train AUC=0.6822, Valid AUC=0.6371, Train Prob Mean=0.2566, Valid Prob Mean=0.2677, Valid Prob Max=0.6110
Epoch 5, Loss=188.1144, Train AUC=0.6972, Valid AUC=0.6260, Train Prob Mean=0.2606, Valid Prob Mean=0.2656, Valid Prob Max=0.7389
Epoch 6, Loss=185.0680, Train AUC=0.7149, Valid AUC=0.6109, Train Prob Mean=0.2694, Valid Prob Mean=0.2716, Valid Prob Max=0.7848
Epoch 7, Loss=180.1028, Train AUC=0.7296, Valid AUC=0.5914, Train Prob Mean=0.2597, Valid Prob Mean=0.2553, Valid Prob Max=0.8732
Epoch 8, Loss=173.3827, Train AUC=0.7517, Valid AUC=0.5704, Train Prob Mean=0.2636, Valid 

<All keys matched successfully>

In [ ]:
# =========================
# 模型输入梯度重要性检测
# =========================

# 目的：粗略检查哪些输入特征对当前模型输出最敏感。
# 如果少数特征梯度远大于其他特征，可能说明模型被它们主导。
#
# 注意：cuDNN 的 LSTM 在 eval() 模式下不允许 RNN backward。
# 因此这里临时关闭 cuDNN 来做输入梯度检测，避免报错：
# RuntimeError: cudnn RNN backward can only be called in training mode

was_training = model.training
model.eval()

X_batch, y_batch = next(iter(valid_loader))
X_batch = X_batch.to(device)
X_batch = X_batch.clone().detach().requires_grad_(True)

model.zero_grad(set_to_none=True)

# 对 LSTM 输入做梯度解释时，临时禁用 cuDNN，保证 eval 模式也能 backward。
with torch.backends.cudnn.flags(enabled=False):
    logits = model(X_batch)
    score = logits.mean()
    score.backward()

if was_training:
    model.train()
else:
    model.eval()

grad = X_batch.grad.detach().cpu().numpy()

# 对 batch 和 time 两个维度取平均绝对梯度
feature_grad_importance = np.abs(grad).mean(axis=(0, 1))
input_abs_mean = np.abs(X_batch.detach().cpu().numpy()).mean(axis=(0, 1))

grad_report = pd.DataFrame({
    "feature": feature_cols,
    "grad_abs_mean": feature_grad_importance,
    "input_abs_mean": input_abs_mean,
})

grad_report["grad_share"] = grad_report["grad_abs_mean"] / (grad_report["grad_abs_mean"].sum() + 1e-12)
grad_report = grad_report.sort_values("grad_abs_mean", ascending=False)

print("输入梯度重要性最高的特征:")
display(grad_report.head(30))

print("Top 10 梯度占比合计:", grad_report.head(10)["grad_share"].sum())
print("Top 20 梯度占比合计:", grad_report.head(20)["grad_share"].sum())


In [ ]:
# =========================
# 测试集评估
# =========================

from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, classification_report

test_prob, test_label, test_logits = predict_loader(model, test_loader)

# 保证都是一维数组，避免后面 DataFrame / 指标计算形状不一致
test_prob = np.asarray(test_prob).reshape(-1)
test_label = np.asarray(test_label).reshape(-1)
test_logits = np.asarray(test_logits).reshape(-1)

test_auc = roc_auc_score(test_label, test_prob)
test_pred_05 = (test_prob >= 0.5).astype(int)

print("Test AUC:", test_auc)
print("Test ACC@0.5:", accuracy_score(test_label, test_pred_05))
print("Test Precision@0.5:", precision_score(test_label, test_pred_05, zero_division=0))
print("Test Recall@0.5:", recall_score(test_label, test_pred_05, zero_division=0))
print("Test F1@0.5:", f1_score(test_label, test_pred_05, zero_division=0))

print("test_prob 分布：")
print(pd.Series(test_prob).describe())

print("真实为1的预测概率：")
print(pd.Series(test_prob[test_label == 1]).describe())

print("真实为0的预测概率：")
print(pd.Series(test_prob[test_label == 0]).describe())

print("test_logits 分布：")
print(pd.Series(test_logits).describe())


In [ ]:
# =========================
# 整理测试集预测结果
# =========================

# 这个 cell 是后面画图和回测的统一入口。
# 后续不要再使用旧变量 result，统一使用 backtest_result。

test_pred_df = pd.DataFrame({
    "datetime": pd.to_datetime(test_index_seq),
    "y_true": test_label,
    "pred_prob": test_prob,
    "pred_label_05": (test_prob >= 0.5).astype(int),
}).set_index("datetime")

backtest_result = test_df.loc[test_pred_df.index].copy()
backtest_result["pred_prob"] = test_pred_df["pred_prob"]
backtest_result["y_true"] = test_pred_df["y_true"]
backtest_result["pred_label_05"] = test_pred_df["pred_label_05"]

# 用分位数组观察概率越高时，真实上涨比例和未来收益是否更高。
# duplicates='drop' 可以避免预测概率过于集中时 qcut 报错。
backtest_result["prob_group"] = pd.qcut(
    backtest_result["pred_prob"],
    q=10,
    labels=False,
    duplicates="drop",
)

group_analysis = backtest_result.groupby("prob_group", observed=True).agg(
    sample_count=("pred_prob", "size"),
    pred_prob_mean=("pred_prob", "mean"),
    label_mean=("y_true", "mean"),
    future_return_mean=("future_return", "mean"),
    future_return_median=("future_return", "median"),
)

print("backtest_result:", backtest_result.shape)
print("index range:", backtest_result.index.min(), "->", backtest_result.index.max())
print("pred_prob range:", backtest_result["pred_prob"].min(), "->", backtest_result["pred_prob"].max())

backtest_result[["close", "future_return", "label", "y_true", "pred_prob", "pred_label_05"]].head()


In [ ]:
# =========================
# 可视化：预测概率、分层效果、价格叠加
# =========================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(backtest_result["pred_prob"], bins=50, alpha=0.8)
axes[0, 0].set_title("Predicted Probability Distribution")
axes[0, 0].set_xlabel("pred_prob")
axes[0, 0].grid(True)

axes[0, 1].hist(
    backtest_result.loc[backtest_result["y_true"] == 0, "pred_prob"],
    bins=40,
    alpha=0.6,
    label="y=0",
)
axes[0, 1].hist(
    backtest_result.loc[backtest_result["y_true"] == 1, "pred_prob"],
    bins=40,
    alpha=0.6,
    label="y=1",
)
axes[0, 1].set_title("Predicted Probability by True Label")
axes[0, 1].legend()
axes[0, 1].grid(True)

group_analysis["label_mean"].plot(ax=axes[1, 0], marker="o")
axes[1, 0].set_title("Positive Label Ratio by Probability Decile")
axes[1, 0].set_xlabel("Probability Group")
axes[1, 0].set_ylabel("Positive Label Ratio")
axes[1, 0].grid(True)

group_analysis["future_return_mean"].plot(ax=axes[1, 1], marker="o")
axes[1, 1].set_title("Future Return Mean by Probability Decile")
axes[1, 1].set_xlabel("Probability Group")
axes[1, 1].set_ylabel("Future Return Mean")
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

group_analysis


## 快速 pandas 回测

先用 pandas 做一个轻量回测，检查模型信号是否有交易价值；确认信号合理后，再接 vn.py 事件驱动回测。


In [ ]:
# =========================
# 快速 pandas 回测
# =========================

# 用验证/测试概率的分布选择阈值。MVP先用固定阈值，也可以改成分位数阈值。
PANDAS_BT_THRESHOLD = 0.55
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.0005
TOTAL_COST = COMMISSION_RATE + SLIPPAGE_RATE

pandas_bt = backtest_result.copy()
pandas_bt["signal"] = (pandas_bt["pred_prob"] >= PANDAS_BT_THRESHOLD).astype(int)

# 当前K线产生的信号，对应未来收益 future_return。
pandas_bt["raw_strategy_return"] = pandas_bt["signal"] * pandas_bt["future_return"]

# 仓位变化时扣成本：0->1 买入扣一次，1->0 卖出扣一次。
pandas_bt["position_change"] = pandas_bt["signal"].diff().abs().fillna(pandas_bt["signal"])
pandas_bt["trade_cost"] = pandas_bt["position_change"] * TOTAL_COST
pandas_bt["strategy_return"] = pandas_bt["raw_strategy_return"] - pandas_bt["trade_cost"]

pandas_bt["equity"] = (1 + pandas_bt["strategy_return"]).cumprod()
pandas_bt["benchmark_equity"] = (1 + pandas_bt["future_return"]).cumprod()
pandas_bt["drawdown"] = pandas_bt["equity"] / pandas_bt["equity"].cummax() - 1

PERIODS_PER_YEAR = 48 * 242

def calc_sharpe(ret):
    std = ret.std()
    return np.nan if std == 0 else ret.mean() / std * np.sqrt(PERIODS_PER_YEAR)

def calc_sortino(ret):
    downside = ret[ret < 0]
    std = downside.std()
    return np.nan if std == 0 else ret.mean() / std * np.sqrt(PERIODS_PER_YEAR)

pandas_bt_stats = {
    "threshold": PANDAS_BT_THRESHOLD,
    "total_return": pandas_bt["equity"].iloc[-1] - 1,
    "benchmark_return": pandas_bt["benchmark_equity"].iloc[-1] - 1,
    "max_drawdown": pandas_bt["drawdown"].min(),
    "sharpe": calc_sharpe(pandas_bt["strategy_return"]),
    "sortino": calc_sortino(pandas_bt["strategy_return"]),
    "trade_count": pandas_bt["position_change"].sum(),
    "signal_ratio": pandas_bt["signal"].mean(),
}

print(pd.Series(pandas_bt_stats))

plt.figure(figsize=(12, 5))
plt.plot(pandas_bt.index, pandas_bt["equity"], label="Strategy")
plt.plot(pandas_bt.index, pandas_bt["benchmark_equity"], label="Benchmark", alpha=0.7)
plt.title("Pandas MVP Backtest Equity Curve")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(pandas_bt.index, pandas_bt["drawdown"], color="red", label="Drawdown")
plt.title("Pandas MVP Backtest Drawdown")
plt.grid(True)
plt.legend()
plt.show()


## vn.py 回测

下面把 `backtest_result["pred_prob"]` 接入 VeighNa/vn.py 的 CTA 回测引擎。策略是 long-only：预测概率高于阈值时买入/持有，低于阈值时卖出/空仓，并在 15:00 强制平仓。


In [ ]:
# 如果还没有安装 CTA 策略模块，先运行这一格。
# 安装后建议重启 kernel，再从本节开始运行。
%pip install vnpy_ctastrategy


In [ ]:
from datetime import datetime, time
from pathlib import Path

import pandas as pd

from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.database import get_database
from vnpy.trader.object import BarData

SYMBOL = "688981"
EXCHANGE = Exchange.SSE
VT_SYMBOL = f"{SYMBOL}.{EXCHANGE.value}"

print(VT_SYMBOL)


In [ ]:
# 1) 将 OHLCV 数据写入 vn.py 数据库。
# vn.py 没有单独的 5分钟 Interval 枚举，所以这里仍然用 Interval.MINUTE 表示分钟级K线。

def save_df_to_vnpy_database(df, gateway_name="LOCAL_5MIN"):
    data_df = df.copy()

    if not isinstance(data_df.index, pd.DatetimeIndex):
        if "datetime" in data_df.columns:
            data_df["datetime"] = pd.to_datetime(data_df["datetime"])
            data_df = data_df.set_index("datetime")
        else:
            raise ValueError("df must have a DatetimeIndex or a 'datetime' column")

    num_cols = ["open", "high", "low", "close", "volume", "amount"]
    data_df[num_cols] = data_df[num_cols].apply(pd.to_numeric, errors="coerce")
    data_df = data_df.dropna(subset=["open", "high", "low", "close", "volume"])
    data_df = data_df.sort_index()
    data_df["amount"] = data_df["amount"].fillna(0.0)

    bars = []
    for dt, row in data_df.iterrows():
        bar_dt = dt.to_pydatetime() if isinstance(dt, pd.Timestamp) else dt
        bars.append(
            BarData(
                symbol=SYMBOL,
                exchange=EXCHANGE,
                datetime=bar_dt,
                interval=Interval.MINUTE,
                open_price=float(row["open"]),
                high_price=float(row["high"]),
                low_price=float(row["low"]),
                close_price=float(row["close"]),
                volume=float(row["volume"]),
                turnover=float(row.get("amount", 0.0)),
                open_interest=0,
                gateway_name=gateway_name,
            )
        )

    database = get_database()
    database.save_bar_data(bars)
    return len(bars)

# 用当前 df 入库即可；如果你保留了原始未dropna数据，也可以替换成原始OHLCV表。
saved_count = save_df_to_vnpy_database(df)
print("saved bars:", saved_count)


In [ ]:
# 2) 导出模型预测信号。
# 后续回测只依赖 backtest_result，不再使用旧变量 result。

def export_signal_file(backtest_result, signal_path="mvp_pred_signal.csv", prob_col="pred_prob"):
    signal_df = backtest_result.copy()

    if not isinstance(signal_df.index, pd.DatetimeIndex):
        if "datetime" in signal_df.columns:
            signal_df["datetime"] = pd.to_datetime(signal_df["datetime"])
        else:
            raise ValueError("backtest_result must have a DatetimeIndex or a 'datetime' column")
    else:
        signal_df["datetime"] = signal_df.index

    signal_df = signal_df[["datetime", prob_col]].dropna().copy()
    signal_df["datetime"] = pd.to_datetime(signal_df["datetime"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    signal_df = signal_df.rename(columns={prob_col: "pred_prob"})

    signal_path = Path(signal_path)
    signal_df.to_csv(signal_path, index=False)
    return signal_path, signal_df

signal_path, signal_df = export_signal_file(backtest_result, "mvp_pred_signal.csv")
print(signal_path)
signal_df.head()


In [ ]:
# 3) 定义 vn.py CTA 策略：根据 pred_prob 做多/空仓切换。
from vnpy_ctastrategy import CtaTemplate
from vnpy_ctastrategy.backtesting import BacktestingEngine

class MLSignalLongOnlyStrategy(CtaTemplate):
    author = "mvp"

    signal_file = "mvp_pred_signal.csv"
    threshold = 0.55
    fixed_size = 100
    exit_at_close = True
    stop_loss_pct = 0.02

    parameters = ["signal_file", "threshold", "fixed_size", "exit_at_close", "stop_loss_pct"]
    variables = ["entry_price"]

    def __init__(self, cta_engine, strategy_name, vt_symbol, setting):
        super().__init__(cta_engine, strategy_name, vt_symbol, setting)
        self.signal_map = {}
        self.entry_price = 0.0

    def on_init(self):
        signal = pd.read_csv(self.signal_file)
        signal["datetime"] = pd.to_datetime(signal["datetime"])
        self.signal_map = dict(zip(signal["datetime"].dt.to_pydatetime(), signal["pred_prob"]))
        self.write_log(f"Loaded {len(self.signal_map)} ML signals")

    def on_start(self):
        self.write_log("Strategy started")

    def on_stop(self):
        self.write_log("Strategy stopped")

    def on_bar(self, bar: BarData):
        self.cancel_all()

        # A股 T+0 研究模拟：尾盘强制平仓，不隔夜。
        if self.exit_at_close and bar.datetime.time() >= time(15, 0):
            if self.pos > 0:
                self.sell(bar.close_price * 0.99, abs(self.pos))
            return

        # 简单止损。
        if self.pos > 0 and self.entry_price > 0:
            if bar.close_price / self.entry_price - 1 <= -self.stop_loss_pct:
                self.sell(bar.close_price * 0.99, abs(self.pos))
                return

        prob = self.signal_map.get(bar.datetime.replace(tzinfo=None))
        if prob is None:
            return

        if prob >= self.threshold and self.pos <= 0:
            self.buy(bar.close_price * 1.01, self.fixed_size)
            self.entry_price = bar.close_price
        elif prob < self.threshold and self.pos > 0:
            self.sell(bar.close_price * 0.99, abs(self.pos))
            self.entry_price = 0.0


In [ ]:
# 4) 运行 vn.py 回测。
# rate=0.001 表示 0.1% 佣金；slippage=0.05 表示每股 0.05 元滑点。

VNPY_BT_THRESHOLD = 0.55

engine = BacktestingEngine()
engine.set_parameters(
    vt_symbol=VT_SYMBOL,
    interval=Interval.MINUTE,
    start=backtest_result.index.min().to_pydatetime(),
    end=backtest_result.index.max().to_pydatetime(),
    rate=0.001,
    slippage=0.05,
    size=1,
    pricetick=0.01,
    capital=1_000_000,
)

engine.add_strategy(
    MLSignalLongOnlyStrategy,
    {
        "signal_file": str(signal_path),
        "threshold": VNPY_BT_THRESHOLD,
        "fixed_size": 100,
        "exit_at_close": True,
        "stop_loss_pct": 0.02,
    },
)

engine.load_data()
engine.run_backtesting()
daily_result = engine.calculate_result()
stats = engine.calculate_statistics(output=False)

pd.Series(stats)


In [ ]:
# 5) 查看 vn.py 回测成交、逐日结果和资金曲线。
trades = engine.get_all_trades()
orders = engine.get_all_orders()

print("trades:", len(trades))
print("orders:", len(orders))

if daily_result is not None and not daily_result.empty:
    display(daily_result.tail())
else:
    print("daily_result is empty")


In [ ]:
# 6) 画 vn.py 回测资金曲线和回撤。
import matplotlib.pyplot as plt

if daily_result is not None and not daily_result.empty and "balance" in daily_result.columns:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(daily_result.index, daily_result["balance"], label="vn.py Strategy Balance")
    axes[0].set_title("vn.py MVP Backtest Balance")
    axes[0].grid(True)
    axes[0].legend()

    if "drawdown" in daily_result.columns:
        axes[1].plot(daily_result.index, daily_result["drawdown"], color="red", label="Drawdown")
        axes[1].set_title("vn.py Drawdown")
        axes[1].grid(True)
        axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("daily_result is empty or has no balance column")
